# Eurovision — воспроизводимая сборка данных

Nikita Nazarov · Ivan Tanaev

Ноутбук повторяет сборку данных, использованных в экспериментах A/B/C.
Он **парсит исходные JSON, HTML, CSV, GeoJSON и сохранённые текстовые таблицы**,
вычисляет признаки и создаёт `eurovision_features.xlsx` с шестью листами.
Готовый Excel и готовые модельные таблицы не являются входами.

**Запуск:** откройте ноутбук из клонированного репозитория и выполните Run all.
Исходники читаются из `data/sources/`, сборка сохраняется в `build/data/`.

## Источники и границы воспроизводимости

| Блок | Исходники | Что вычисляем |
|---|---|---|
| Результаты и национальные баллы | EurovisionAPI, зафиксированный Git commit | Состав, места, цели A/B/C |
| Контроль результатов | Сохранённые официальные Scoreboard и HTML таблиц Wikipedia | Сверка мест и каждого национального балла |
| Песня | Опубликованные CSV Song Science | Язык, BPM, danceability, динамика, слова, повторяемость |
| Порядок выступления | Датированные публикации, сохранённые страницы и реестр извлечения | Исходный слот и относительная позиция |
| История | Результаты 2003–2025 и национальные голоса с 2016 | Только предыдущие пять календарных лет |
| География | Natural Earth v5.1.2 | Расстояние между столицами, log1p расстояния, сухопутная граница |
| Рынок | Сохранённые таблицы Eurovisionworld, 2015–2025 без 2020 | Нормированные котировки пяти букмекеров |

Это **воспроизводимая сборка из зафиксированных исходников**, а не обещание заново
получить историческое состояние каждого сайта. Официальные Scoreboard и часть
публикаций получены через браузер; их снимки входят в `data/sources/`. Парсер этих снимков
приведён ниже. Опциональный режим `download` повторяет HTTP-загрузку доступных
машиночитаемых источников и проверяет точное совпадение SHA-256.

Аудиопризнаки берём у авторов Song Science; из аудиозаписей здесь их не извлекаем.
**Время измерения архивных котировок неизвестно.** Время получения файла не доказывает
доступность котировок до финала. Этот ноутбук не устраняет указанное ограничение.

In [ ]:
# В Colab зависимости обычно уже установлены. При необходимости выполните:
# %pip install "pandas>=2.2,<4" "openpyxl>=3.1,<4"

from pathlib import Path
import hashlib
import importlib
import json
import platform
import shutil
import subprocess
import sys
import zipfile
import numpy as np
import pandas as pd
import openpyxl
from IPython.display import display, FileLink

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p / "data/eurovision_features.xlsx").is_file())
SOURCE_DIR = ROOT / "data/sources"
BUILD_DIR = ROOT / "build/data"
SOURCE_MODE = "frozen"  # frozen / download
ALLOW_SNAPSHOT_FALLBACK = True
assert SOURCE_MODE in {"frozen", "download"}
BUILD_DIR.mkdir(parents=True, exist_ok=True)
(BUILD_DIR / "scripts").mkdir(exist_ok=True)
print("Python:", platform.python_version(), "pandas:", pd.__version__, "openpyxl:", openpyxl.__version__)
print("Рабочая папка:", BUILD_DIR)

## 1. Зафиксированные исходники

Проверяем каждый путь и SHA-256 каждого исходного файла перед копированием.
Все операции выполняются в отдельной папке `build/data/`.
`reproduction_manifest.json` содержит URL, даты получения, тип снимка и хеши.

In [ ]:
def file_sha256(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()

manifest = json.loads((SOURCE_DIR / "reproduction_manifest.json").read_text())
for item in manifest["files"]:
    src = (SOURCE_DIR / item["file"]).resolve()
    dst = (BUILD_DIR / item["file"]).resolve()
    assert src.is_relative_to(SOURCE_DIR.resolve())
    assert dst.is_relative_to(BUILD_DIR.resolve())
    assert src.stat().st_size == item["bytes"]
    assert file_sha256(src) == item["sha256"], item["file"]
    dst.parent.mkdir(parents=True, exist_ok=True)
    shutil.copyfile(src, dst)
shutil.copyfile(SOURCE_DIR / "reproduction_manifest.json", BUILD_DIR / "reproduction_manifest.json")
source_catalog = pd.DataFrame(manifest["files"])
display(source_catalog.groupby("kind").agg(files=("file", "size"), bytes=("bytes", "sum")))
print("Verified source files:", len(source_catalog))


## 2. Повторная загрузка из интернета — опционально

При `SOURCE_MODE = "frozen"` сеть не нужна. При `download` заново запрашиваем
источники типа `downloadable`: API JSON, опубликованные CSV, географию и HTML.
Принимаем только байты с исходным SHA-256. Иное содержимое сайта не подменяет датасет.
Браузерные и текстовые снимки берём из `data/sources/` в обоих режимах.

Каждая ошибка HTTP или несовпадение хеша попадает в `source_acquisition.csv`.
Если `ALLOW_SNAPSHOT_FALLBACK=False`, любая такая ошибка останавливает сборку.

In [ ]:
import concurrent.futures
import time
import urllib.request
from datetime import datetime, timezone

def acquire_source(item):
    record = dict(file=item["file"], url=item["url"], kind=item["kind"],
                  status="frozen", error="", checked_utc="")
    if SOURCE_MODE != "download" or item["kind"] != "downloadable":
        return record
    record["checked_utc"] = datetime.now(timezone.utc).isoformat()
    error = None
    for attempt in range(2):
        try:
            request = urllib.request.Request(item["url"], headers={"User-Agent": "Eurovision-academic-reproduction/1.0"})
            with urllib.request.urlopen(request, timeout=30) as response:
                body = response.read()
                record["resolved_url"] = response.geturl()
            if hashlib.sha256(body).hexdigest() != item["sha256"]:
                raise ValueError("SHA-256 differs from the original source snapshot")
            (BUILD_DIR / item["file"]).write_bytes(body)
            record["status"] = "downloaded_verified"
            return record
        except Exception as exc:
            error = exc
            if isinstance(exc, ValueError):
                break
            if attempt == 0:
                time.sleep(1)
    record.update(status="fallback_to_frozen", error=f"{type(error).__name__}: {error}")
    return record

with concurrent.futures.ThreadPoolExecutor(max_workers=6) as pool:
    acquisition = pd.DataFrame(list(pool.map(acquire_source, manifest["files"])))
acquisition.to_csv(BUILD_DIR / "source_acquisition.csv", index=False)
display(acquisition.groupby(["kind", "status"]).size().rename("files").reset_index())
failures = acquisition.loc[acquisition.status.eq("fallback_to_frozen")]
if len(failures):
    display(failures[["file", "error"]])
    if not ALLOW_SNAPSHOT_FALLBACK:
        raise RuntimeError("Повторная загрузка неполна. См. source_acquisition.csv.")
    print("Использованы проверенные исходные снимки для", len(failures), "недоступных/изменённых файлов.")

## 3. Парсеры и правила сборки

Следующие четыре ячейки содержат полный код использованного конвейера.
`%%writefile` сохраняет его как обычные Python-модули в рабочую папку.
Их можно читать, менять и запускать отдельно; внешнего пакета с нашими парсерами не требуется.

### 3.1. Финалисты, результаты, признаки песни

Читаем исходные CSV и JSON, разбираем официальный Scoreboard и контрольные HTML-таблицы. Восстанавливаем отдельные места зрителей/жюри с разрешением ничьих. `target_top10 = place_final ≤ 10`; цель B — `place_tele < place_jury`. Состав 2024 исключает NL; 2020 отсутствует.

In [ ]:
%%writefile {BUILD_DIR}/scripts/build_entries.py
#!/usr/bin/env python3
"""Build and validate the Eurovision entries base table, offline, with Python 3.10+."""
import argparse
import collections
import csv
import hashlib
import html
import json
import math
import re
from pathlib import Path

ROOT = Path(__file__).resolve().parents[1]
RAW = ROOT / 'data/raw/entries'
SAMPLES = ROOT / 'research/source_samples'
OUT = ROOT / 'data/processed'
REPORT = ROOT / 'reports/entries'
CONFIG = ROOT / 'data/config'
YEARS = [y for y in range(2008, 2026) if y != 2020]
FEATURE_MAP = {
    'english_only': 'lyrics_english', 'english_mixed': 'lyrics_english_mix',
    'bpm': 'bpm', 'danceability': 'danceability',
    'dynamic_complexity': 'dynamic_complexity', 'total_words': 'total_words',
    'repetitiveness': 'n_gram_repetitiveness',
}
CORE_FEATURES = ['running_position', 'host', 'english_only', 'english_mixed',
    'bpm', 'danceability', 'dynamic_complexity', 'log_word_count', 'repetitiveness']
ALIASES = {'Czech Republic': 'Czechia', 'Macedonia': 'North Macedonia',
    'F.Y.R. Macedonia': 'North Macedonia', 'Bosnia and Herzegovina': 'Bosnia & Herzegovina',
    'The Netherlands': 'Netherlands', 'Türkiye': 'Turkey'}


def read_csv(path):
    with path.open(encoding='utf-8-sig', newline='') as stream:
        return list(csv.DictReader(stream))


def dump_csv(path, rows, fields=None):
    path.parent.mkdir(parents=True, exist_ok=True)
    if fields is None:
        fields = list(rows[0]) if rows else []
    with path.open('w', encoding='utf-8', newline='') as stream:
        writer = csv.DictWriter(stream, fieldnames=fields, lineterminator='\n')
        writer.writeheader()
        writer.writerows(rows)


def load_json(path):
    return json.loads(path.read_text(encoding='utf-8'))


def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()


def numeric(value, integer=False):
    if value is None or str(value).strip().lower() in {'', 'nan', 'none', 'null'}:
        return None
    number = float(value)
    if not math.isfinite(number):
        raise ValueError(f'Nonfinite numeric input: {value}')
    if integer:
        if number != int(number):
            raise ValueError(f'Expected integer, got {value}')
        return int(number)
    return number


def text_cell(value):
    value = re.sub(r'<sup\b[^>]*>.*?</sup>', '', value, flags=re.S)
    return ' '.join(html.unescape(re.sub(r'<[^>]*>', '', value)).split())


def parse_split_validation(path, n, name_to_code):
    """Read the explicit Split results table, never derive validation ranks from points."""
    tables = re.findall(r'<table\b[^>]*>.*?</table>', path.read_text(), re.S)
    matches = []
    for table in tables:
        if not all(x in table for x in ['Combined', 'Jury', 'Televoting']):
            continue
        records = []
        for row in re.findall(r'<tr\b[^>]*>(.*?)</tr>', table, re.S):
            cells = [text_cell(c) for c in re.findall(r'<t[dh]\b[^>]*>(.*?)</t[dh]>', row, re.S)]
            if len(cells) == 7 and cells[0].isdigit():
                records.append(cells)
        if len(records) == n:
            matches.append(records)
    if len(matches) != 1:
        raise ValueError(f'{path}: expected one complete split ranking table, got {len(matches)}')
    result = {}
    for cells in matches[0]:
        rank = int(cells[0])
        for channel, start in [('total', 1), ('jury', 3), ('public', 5)]:
            name = re.sub(r'\[[^\]]+\]', '', cells[start]).strip()
            code = name_to_code[ALIASES.get(name, name)]
            key = (code, channel)
            if key in result:
                raise ValueError(f'Duplicate validation rank: {key}')
            result[key] = (rank, int(cells[start+1]))
    return result


def split_ranks(performances, channel):
    """Total points, number of supporting voters, countback 12..1, earlier slot."""
    keys = []
    for performance in performances:
        score = next(s for s in performance['scores'] if s['name'] == channel)
        votes = list(score['votes'].values())
        key = (-score['points'], -sum(v > 0 for v in votes),
            *[-votes.count(p) for p in [12, 10, 8, 7, 6, 5, 4, 3, 2, 1]],
            performance['running'])
        keys.append((key, performance['contestantId']))
    if len({key for key, _ in keys}) != len(keys):
        raise ValueError('Unresolved split ranking tie')
    return {contestant_id: rank for rank, (_, contestant_id) in enumerate(sorted(keys), 1)}


def build(output=OUT, report_dir=REPORT):
    decisions = load_json(CONFIG / 'entries_decisions.json')
    names = {r['country']: r['country_name'] for r in read_csv(CONFIG/'countries.csv')}
    codes = {name: code for code, name in names.items()}
    checks, comparisons, issues, provenance, ties = [], [], [], [], []
    inputs = set()

    def check(name, passed, detail, year=None):
        checks.append(dict(year=year, check=name, status='pass' if passed else 'fail', detail=detail))

    def compare(year, country, field, left_source, left, right_source, right, allowed=False):
        status = 'match' if left == right else ('explained' if allowed else 'mismatch')
        comparisons.append(dict(year=year, country=country, field=field,
            source_a=left_source, value_a=left, source_b=right_source, value_b=right, status=status))

    original_manifest = {r['file']: r for r in load_json(ROOT/'research/source_manifest.json')}
    api_manifest = {r['file']: r for r in load_json(RAW/'manifest.json')}
    for filename in ['finalists.csv', 'songs2025.csv', 'song-readme.md']:
        path = SAMPLES/filename
        check('source_hash', sha256(path) == original_manifest[filename]['sha256'], str(path.relative_to(ROOT)))
        inputs.add(path)
    features = {}
    source_rows = {}
    for filename in ['finalists.csv', 'songs2025.csv']:
        for line, row in enumerate(read_csv(SAMPLES/filename), 1):
            key = (int(row['year']), codes[row['to_country']])
            if key in features:
                raise ValueError(f'Duplicate feature key: {key}')
            features[key] = row
            source_rows[key] = (filename, line)
    entries, coverage = [], []
    for year in YEARS:
        directory = RAW/f'api/{year}'
        final = load_json(directory/'rounds/final.json')
        contest = load_json(directory/'contest.json')
        official_path = RAW/f'official_browser/{year}.json'
        official = load_json(official_path)
        inputs.add(official_path)
        official_rows = {}
        for row in official['rows']:
            name = row['label'].removeprefix('Scoreboard entry for ')
            code = codes[ALIASES.get(name, name)]
            if code in official_rows:
                raise ValueError(f'Duplicate official key {year} {code}')
            parts = [s.strip() for s in row['text'].splitlines() if s.strip()]
            parsed = dict(country=code, place=int(parts[0]),
                total=int(re.search(r'(\d+) points', row['text'])[1]),
                running=int(re.search(r'Running Order\s+(\d+)', row['text'])[1]))
            for channel, label in [('jury', 'Jury'), ('public', '(?:Audience|Televote)')]:
                match = re.search(label+r'\s+(\d+)', row['text'])
                parsed[channel] = int(match[1]) if match else None
            official_rows[code] = parsed
        metadata = {}
        for path in sorted((directory/'contestants').glob('*.json')):
            c = load_json(path)
            if c['id'] in metadata:
                raise ValueError(f'Duplicate API id {year} {c["id"]}')
            metadata[c['id']] = (c, path)
        performances = final['performances']
        n = len(performances)
        api_countries = {metadata[p['contestantId']][0]['country'] for p in performances}
        check('finalist_count', n == decisions['expected_finalists'][str(year)], str(n), year)
        check('official_membership', api_countries == set(official_rows), str(sorted(api_countries)), year)
        check('unique_finalist_countries', len(api_countries) == n, str(n), year)
        check('final_places_permutation', sorted(p['place'] for p in performances) == list(range(1,n+1)), str(n), year)
        slots = sorted(p['running'] for p in performances)
        expected_slots = [i for i in range(1,27) if i!=5] if year == 2024 else list(range(1,n+1))
        check('running_slots', slots == expected_slots, str(slots), year)
        dense = {slot: index for index, slot in enumerate(slots, 1)}
        ranks = {ch: split_ranks(performances, ch) for ch in ['jury','public']} if year>=2016 else {}
        validation = {}
        if year>=2016:
            path = RAW/f'validation/{year}.html'
            validation = parse_split_validation(path, n, codes)
            inputs.add(path)
            for ch in ['jury','public']:
                groups = collections.defaultdict(list)
                for p in performances:
                    score = next(s for s in p['scores'] if s['name']==ch)
                    groups[score['points']].append(p)
                for points, group in groups.items():
                    if len(group)>1:
                        for p in sorted(group, key=lambda p:ranks[ch][p['contestantId']]):
                            score = next(s for s in p['scores'] if s['name']==ch)
                            values = list(score['votes'].values())
                            ties.append(dict(year=year, channel=ch, points=points,
                                country=metadata[p['contestantId']][0]['country'],
                                rank=ranks[ch][p['contestantId']], supporting_voters=sum(v>0 for v in values),
                                countback=json.dumps({v:values.count(v) for v in [12,10,8,7,6,5,4,3,2,1]}),
                                running_slot=p['running']))
        year_entries=[]
        for p in performances:
            c, meta_path = metadata[p['contestantId']]
            country = c['country']
            key = (year, country)
            source = features.get(key)
            if source is None:
                raise ValueError(f'Missing Song Science match {key}')
            filename, line = source_rows[key]
            primary = official_rows[country]
            scores = {s['name']: s for s in p['scores']}
            compare(year,country,'place_final','EurovisionAPI',p['place'],'official',primary['place'])
            compare(year,country,'points_final','EurovisionAPI',scores['total']['points'],'official',primary['total'])
            # Current official archive uses the dense sequence for 2024; API preserves the original voting slot.
            compare(year,country,'running_slot_or_sequence','EurovisionAPI',p['running'],'official',primary['running'],
                allowed=year==2024 and primary['running']==dense[p['running']])
            for ch in ['jury','public'] if year>=2016 else []:
                compare(year,country,'points_'+ch,'EurovisionAPI',scores[ch]['points'],'official',primary[ch])
                compare(year,country,'place_'+ch,'computed_tiebreak',ranks[ch][c['id']],
                    'independent_split_table',validation[(country,ch)][0])
                compare(year,country,'points_'+ch,'EurovisionAPI',scores[ch]['points'],
                    'independent_split_table',validation[(country,ch)][1])
                check('vote_sum_'+ch, sum(scores[ch]['votes'].values())==scores[ch]['points'], country, year)
                check('national_point_values_'+ch, all(v in {0,1,2,3,4,5,6,7,8,10,12} for v in scores[ch]['votes'].values()), country, year)
                check('no_self_vote_'+ch, scores[ch]['votes'].get(country,0)==0, country, year)
            if year>=2016:
                check('jury_plus_public', scores['jury']['points']+scores['public']['points']==scores['total']['points'], country, year)
            for field, actual in [('place_final',p['place']),('running_final',p['running']),
                    ('points_final',scores['total']['points'])]+[(f'points_{ch}_final',scores[api]['points'])
                    for ch,api in [('tele','public'),('jury','jury')] if api in scores]:
                if source.get(field):
                    compare(year,country,field,'Song Science',numeric(source[field],True),'EurovisionAPI',actual)
            e=dict(year=year,country=country,country_name=names[country],artist=c['artist'],song=c['song'],
                final_date=final['date'],cutoff_utc=final['date']+'T12:00:00+00:00',
                n_finalists=n,in_cutoff_final=1,running_slot=p['running'],running_order=dense[p['running']],
                running_position=(dense[p['running']]-1)/(n-1),host=int(contest['country']==country),
                place_final=p['place'],points_final=scores['total']['points'],
                place_tele=ranks.get('public',{}).get(c['id']),place_jury=ranks.get('jury',{}).get(c['id']),
                points_tele=scores.get('public',{}).get('points'),points_jury=scores.get('jury',{}).get('points'),
                target_top10=int(p['place']<=10),
                target_audience_above_jury=int(ranks['public'][c['id']]<ranks['jury'][c['id']]) if ranks else None)
            for field, source_column in FEATURE_MAP.items():
                e[field]=numeric(source.get(source_column),field in {'english_only','english_mixed','total_words'})
            e['log_word_count']=math.log1p(e['total_words']) if e['total_words'] is not None else None
            e['language_group']=('english' if e['english_only']==1 else 'mixed_english' if e['english_mixed']==1
                else 'no_english') if e['english_only'] is not None and e['english_mixed'] is not None else None
            e.update(lyrics_language_raw=source.get('lyrics_language',''),
                audio_duration_seconds=numeric(source.get('length(s)')),audio_url=source.get('audio_url',''),
                audio_url2=source.get('audio_url2',''),pre_contest_url=source.get('pre_contest_url',''),
                source_artist=source['performer'],source_song=source['song'],
                results_status='official_totals_checked',split_rank_status='tiebreak_reconstructed_and_independently_checked' if ranks else 'not_applicable',
                audio_status='published_studio_features_accepted_by_user',
                cutoff_membership_status='dated_prefinal_lineup_checked_exclusion_timed' if year==2024 else 'dated_prefinal_lineup_and_order_checked',
                core_training_ready=1,
                feature_source_file='research/source_samples/'+filename,feature_source_record=line,
                feature_source_url=original_manifest[filename]['url'],
                result_source_file=str((directory/'rounds/final.json').relative_to(ROOT)),
                result_source_url=api_manifest[f'api/{year}/rounds/final.json']['url'],
                official_source_file=str(official_path.relative_to(ROOT)),official_source_url=official['url'],
                identity_source_file=str(meta_path.relative_to(ROOT)))
            check('language_indicators', e['english_only'] in {0,1} and e['english_mixed'] in {0,1}
                and e['english_only']+e['english_mixed']<=1,country,year)
            check('raw_numeric_features', all(e[f] is not None for f in FEATURE_MAP),country,year)
            check('numeric_ranges', e['bpm']>0 and e['danceability']>=0 and e['dynamic_complexity']>=0
                and e['total_words']>=0 and 0<=e['repetitiveness']<=1,country,year)
            if e['audio_duration_seconds']>230:
                issues.append(dict(year=year,country=country,code='LONG_STUDIO_RECORDING',status='accepted_source_features',
                    field='audio_duration_seconds',detail=f"{e['audio_duration_seconds']} seconds; published source features accepted by user, including full studio/video versions. No independent audio reconstruction claimed.",source=e['audio_url']))
            if source['performer'].casefold()!=c['artist'].casefold():
                issues.append(dict(year=year,country=country,code='ARTIST_ALIAS',status='resolved',field='artist',
                    detail=f"Song Science: {source['performer']}; canonical API stage name: {c['artist']}. Join uses country/year, not names.",source=e['feature_source_url']))
            for field in e:
                if field in FEATURE_MAP or field in {'audio_duration_seconds','audio_url','audio_url2','pre_contest_url','lyrics_language_raw','source_artist','source_song'}:
                    source_file=e['feature_source_file'];record=f'CSV data record {line}';source_url=e['feature_source_url'];method=FEATURE_MAP.get(field,field)
                elif field in {'place_final','points_final','points_tele','points_jury','running_slot','final_date','place_tele','place_jury','target_top10','target_audience_above_jury'}:
                    source_file=e['result_source_file'];record=f'contestantId={c["id"]}';source_url=e['result_source_url'];method='split_tiebreak' if field in {'place_tele','place_jury'} else 'derived' if field.startswith('target_') else 'direct'
                elif field in {'artist','song','country'}:
                    source_file=str(meta_path.relative_to(ROOT));record=f'id={c["id"]}'
                    source_url=api_manifest[str(meta_path.relative_to(RAW))]['url'];method='direct'
                elif field in {'running_order','running_position','n_finalists','in_cutoff_final','cutoff_utc'}:
                    source_file=e['result_source_file'];record=f'contestantId={c["id"]}'
                    source_url=e['result_source_url'];method=field+'; data/config/entries_decisions.json'
                elif field=='host':
                    source_file=str((directory/'contest.json').relative_to(ROOT));record='country'
                    source_url=api_manifest[f'api/{year}/contest.json']['url'];method='country == host country'
                elif field in {'log_word_count','language_group'}:
                    source_file=e['feature_source_file'];record=f'CSV data record {line}';source_url=e['feature_source_url']
                    method='log1p(total_words)' if field=='log_word_count' else 'english_only / english_mixed indicators'
                else:
                    continue
                provenance.append(dict(year=year,country=country,field=field,source_file=source_file,
                    source_record=record,source_url=source_url,method=method))
            year_entries.append(e)
            inputs.add(meta_path)
        check('top10_capacity',sum(e['target_top10'] for e in year_entries)==10,'exactly ten',year)
        check('position_range',all(0<=e['running_position']<=1 for e in year_entries),'[0,1]',year)
        check('exactly_one_host',sum(e['host'] for e in year_entries)==1,contest['country'],year)
        for ch,column in [('jury','place_jury'),('public','place_tele')] if year>=2016 else []:
            check('split_permutation_'+ch,sorted(e[column] for e in year_entries)==list(range(1,n+1)),str(n),year)
        entries.extend(year_entries)
        coverage.append(dict(year=year,expected_finalists=n,assembled_finalists=len(year_entries),
            matched_features=len(year_entries),top10_labels=sum(e['target_top10'] is not None for e in year_entries),
            split_labels=sum(e['target_audience_above_jury'] is not None for e in year_entries),
            complete_common_features=sum(all(e[f] is not None for f in CORE_FEATURES) for e in year_entries),
            official_results_checked=len(official_rows),split_ranks_independently_checked=n if ranks else 0,
            audio_version_date_verified=0,core_training_ready=n,
            status='complete_published_audio_features_accepted'))
        inputs.update([directory/'contest.json',directory/'rounds/final.json'])
    # Verify pinned cache bytes for every downloaded file used in this build.
    for path in sorted(inputs):
        relative=str(path.relative_to(RAW)) if path.is_relative_to(RAW) else None
        if relative in api_manifest:
            check('downloaded_source_hash',sha256(path)==api_manifest[relative]['sha256'],relative)
    mismatches=[r for r in comparisons if r['status']=='mismatch']
    check('source_reconciliation',not mismatches,f'{len(mismatches)} unexplained mismatches')
    check('unique_entry_keys',len({(e['year'],e['country']) for e in entries})==len(entries),str(len(entries)))
    exclusions=[dict(year=2020,country='ALL',task='A/B',reason='Contest cancelled; no outcome',
        known_before_cutoff='yes',source='https://www.eurovision.com/stories/eurovision-2020-in-rotterdam-is-cancelled/') ,
        dict(year=2024,country='NL',task='A/B',reason='Disqualified before cutoff; not a negative top10 observation',
        known_before_cutoff='yes; known by 10:19 UTC',source=decisions['sources_reviewed'][2]['timing_evidence_url'])]
    from build_final_inputs import validate_running_orders
    running_sources=validate_running_orders(entries)
    check('dated_prefinal_running_orders',len(running_sources)==len(entries),'All 17 dated publications match all finalist slots')
    inputs.add(CONFIG/'running_order_evidence.json')
    inputs.update(ROOT/e['source_file'] for e in load_json(CONFIG/'running_order_evidence.json'))
    for year in YEARS:
        issues.append(dict(year=year,country='ALL',code='AUDIO_VERSION_DATE',status='accepted_source_policy',field='audio',
            detail='User accepts published Song Science features and author-declared studio origin. Independent per-track date/version audit waived, not claimed completed.',
            source=original_manifest['song-readme.md']['url']))
    feature_coverage=[]
    for year in YEARS:
        group=[e for e in entries if e['year']==year]
        for field in CORE_FEATURES+['place_final','place_tele','place_jury','target_top10','target_audience_above_jury','audio_duration_seconds']:
            vals=[e[field] for e in group if e[field] is not None]
            feature_coverage.append(dict(year=year,field=field,rows=len(group),present=len(vals),missing=len(group)-len(vals),
                applicability='not_applicable' if year<2016 and field in {'place_tele','place_jury','target_audience_above_jury'} else 'applicable',
                min=min(vals) if vals else None,max=max(vals) if vals else None))
    entries.sort(key=lambda e:(e['year'],e['country']))
    for filename,rows in [('checks.csv',checks),('source_comparisons.csv',comparisons),('issues.csv',issues),
            ('coverage.csv',coverage),('feature_coverage.csv',feature_coverage),('exclusions.csv',exclusions),
            ('ties.csv',ties),('provenance.csv',provenance)]:
        dump_csv(report_dir/filename,rows)
    failures=[c for c in checks if c['status']=='fail']
    # Never overwrite a usable entries.csv with a build that failed validation.
    if failures:
        raise ValueError(f'{len(failures)} failed checks; inspect {report_dir}/checks.csv and source_comparisons.csv')
    dump_csv(output/'entries.csv',entries)
    inputs.update([CONFIG/'entries_decisions.json',CONFIG/'countries.csv',ROOT/'research/source_manifest.json',
        RAW/'manifest.json',RAW/'repository_lock.json',Path(__file__).resolve()])
    source_fingerprints=[dict(file=str(p.relative_to(ROOT)),sha256=sha256(p)) for p in sorted(inputs)]
    (report_dir/'build_manifest.json').write_text(json.dumps(dict(
        rows=len(entries),years=YEARS,entries_sha256=sha256(output/'entries.csv'),checks_passed=len(checks),
        api_commit=load_json(RAW/'repository_lock.json')['commit'],input_fingerprints=source_fingerprints),indent=2)+'\n')
    write_report(report_dir,entries,coverage,checks,comparisons,issues,ties)
    print(f'Built {len(entries)} entries; {len(checks)} checks passed; {len(mismatches)} unexplained source mismatches.')
    print(f'Split labels: {sum(e["target_audience_above_jury"] is not None for e in entries)}. Core ready under accepted published-feature policy: {len(entries)}.')
    return entries


def write_report(directory, entries, coverage, checks, comparisons, issues, ties):
    lines=['# Сборка entries: результат проверки','',
        f'Собрано **{len(entries)} финалистов из 17 конкурсов 2008–2025**, без 2020. '
        f'Для задачи B получено **{sum(e["target_audience_above_jury"] is not None for e in entries)} метки за 2016–2025**.','',
        'Базовая таблица готова. Готовые признаки Song Science приняты по решению пользователя; '
        'это доверие к опубликованному датасету, а не заявление о независимой проверке каждой аудиоверсии.','',
        '## Покрытие','',
        '| Год | Финалисты | Признаки 9/9 | Метки A | Метки B | Официальная сверка |',
        '|---|---:|---:|---:|---:|---:|']
    for c in coverage:
        lines.append(f'| {c["year"]} | {c["assembled_finalists"]} | {c["complete_common_features"]} | {c["top10_labels"]} | {c["split_labels"]} | {c["official_results_checked"]} |')
    long_audio=[i for i in issues if i['code']=='LONG_STUDIO_RECORDING']
    lines += ['', '## Проверено','',
        f'- {len(checks)} автоматических проверок прошли. Необъяснённых расхождений источников: 0.',
        '- Ключ `(year, country)` уникален. Состав сверён с официальной таблицей каждого финала.',
        '- Итоговые места образуют перестановку 1…N; в каждом году ровно десять меток top10.',
        '- Места жюри и зрителей восстановлены с разрешением ничьих и сверены с отдельной таблицей Split results за каждый год. '
        'Таблица Wikipedia служит независимой проверкой, первичные баллы взяты из EurovisionAPI и сверены с официальными итогами.',
        '- В раздельных голосованиях проверены суммы национальных баллов, допустимые значения, отсутствие самоголосования и равенство total = jury + public.',
        '- Все девять общих признаков заполнены. Danceability не ограничивается единицей.',
        '- Готовые LY-агрегаты, просмотры YouTube, результаты полуфиналов, технические индексы и contestantId в признаки не включены.','',
        '## Разобранные случаи','',
        '- **2019:** использована исправленная версия результатов. Итоги согласуются с [заявлением EBU от 22 мая](https://www.ebu.ch/news/2019/05/ebu-issues-statement-on-eurovision-song-contest-grand-final-jury-result).',
        '- **2024:** исключение NL известно не позже 10:19 UTC, до cutoff 12:00 UTC '
        '([хроника NOS, сообщение 12:19 CEST](https://nos.nl/collectie/13971/liveblog/2520041-winnaar-nemo-vooral-bij-vakjury-populair-joost-klein-tweede-in-halve-finale)). '
        'Год сохранён, NL не получает искусственную нулевую метку.',
        '- **Порядок 2024:** `running_slot` сохраняет номера 1–26 с пропуском 5; `running_order` — фактическую последовательность 1–25. '
        '`running_position = (running_order-1)/(N-1)`. Новый официальный архив использует плотную последовательность, '
        'предфинальная [публикация порядка](https://www.eurovision.com/stories/eurovision-2024-the-grand-final-running-order/) — исходные номера. '
        'Это объяснённое различие, а не незаметное исправление исходника.',
        '- **2023:** признак host относится к Великобритании, где проходил конкурс, а не к предыдущему победителю Украине.',
        '- **Ничьи:** см. `ties.csv`. Сортировка включает количество поддержавших голосующих, countback и порядок выступления. '
        'Все полученные места сверены независимо. Нулевые баллы сохранены как наблюдения.',
        '- **Имена:** пять различий между полным именем и сценическим именем сохранены в `issues.csv`; объединение не зависит от написания имени.','',
        '## Принятые решения','',
        f'1. Используем опубликованные признаки песен без пересчёта. У {len(long_audio)} записей длина превышает 230 секунд; '
        'сохраняем версии источника. Статус `published_studio_features_accepted_by_user`, `core_training_ready=1`. '
        'Индивидуальная проверка даты/версии аудио не заявляется выполненной.',
        '2. Все 437 номеров и состав всех 17 финалов сверены с датированными предфинальными публикациями. '
        'Источники: `data/config/running_order_evidence.json`, подробный итог: `reports/final_inputs/README.md`. Особый случай 2024 разобран отдельно.',
        '3. Исторические признаки, national_votes и география собираются отдельно командой `python3 scripts/build_remaining.py`; '
        'готовые результаты описаны в `reports/remaining/README.md`. В исходную entries эти дополнительные столбцы не подмешаны. Odds — отдельный блок.','',
        '## Воспроизведение','',
        'Из корня проекта: `python3 scripts/build_entries.py`. Нужен только Python 3.10+, сеть и дополнительные пакеты не нужны. '
        'При ошибке проверки скрипт завершается с ненулевым кодом и не заменяет `entries.csv`.',
        'Снимки официальных таблиц сохранены из браузера в `data/raw/entries/official_browser`. '
        'Прямые HTTP-запросы возвращали 403; журнал попыток сохранён. '
        'Браузерная выгрузка читает раскрытые видимые строки Scoreboard, не скрытое состояние приложения.',
        'Все использованные исходники, их SHA-256 и commit EurovisionAPI перечислены в `build_manifest.json`. '
        'Новые сетевые выгрузки выполняет `scripts/fetch_entries_sources.py`; исходные исследовательские копии не изменены.','',
        'Словарь полей: `DATA_DICTIONARY.md`. Подробности: `coverage.csv`, `feature_coverage.csv`, '
        '`source_comparisons.csv`, `provenance.csv`, `exclusions.csv`, `issues.csv`, `checks.csv`.','']
    (directory/'README.md').write_text('\n'.join(lines),encoding='utf-8')


if __name__ == '__main__':
    parser=argparse.ArgumentParser(description=__doc__)
    parser.add_argument('--output-dir',type=Path,default=OUT)
    parser.add_argument('--report-dir',type=Path,default=REPORT)
    args=parser.parse_args()
    build(args.output_dir,args.report_dir)


### 3.2. История стран и пар, национальные голоса, география

Исторические признаки используют только `[год−5, год−1]`, без 2020. Сглаживание — три псевдонаблюдения; число наблюдений — `log1p(n)`. История пары учитывает отклонение баллов от `58 / число кандидатов`. Расстояние — haversine между столицами и `log1p(distance_km)`; граница — явная сухопутная линия Natural Earth. C включает полный набор допустимых кандидатов, без самоголосования, Rest of the World и синтетического San Marino.

In [ ]:
%%writefile {BUILD_DIR}/scripts/build_remaining.py
#!/usr/bin/env python3
"""Build country histories, audited national votes, pair histories, and geography offline."""
import argparse
import collections
import json
import math
import re
from pathlib import Path

from build_entries import (ROOT, YEARS, CORE_FEATURES, ALIASES, read_csv, dump_csv,
    load_json, sha256, numeric, text_cell)

RAW=ROOT/'data/raw/remaining'
ENTRY_RAW=ROOT/'data/raw/entries'
OUT=ROOT/'data/processed'
REPORT=ROOT/'reports/remaining'
DECISIONS=ROOT/'data/config/remaining_decisions.json'
FEATURES_A=CORE_FEATURES+['past_final_strength','past_final_count']
FEATURES_B=CORE_FEATURES+['past_audience_advantage','past_split_count']
FEATURES_C=CORE_FEATURES+['past_pair_support','past_pair_count','log_distance','shared_border']
VOTE_YEARS=[y for y in YEARS if y>=2016]
SCALE=[12,10,8,7,6,5,4,3,2,1]


def clean_name(name):
    name=re.sub(r'\[[^\]]+\]','',name).strip()
    return ALIASES.get(name,name)


def table_rows(table):
    return [[text_cell(cell) for cell in re.findall(r'<t[dh]\b[^>]*>(.*?)</t[dh]>',row,re.S)]
        for row in re.findall(r'<tr\b[^>]*>(.*?)</tr>',table,re.S)]


def parse_history_validation(path,n,codes):
    candidates=[]
    for table in re.findall(r'<table\b[^>]*>.*?</table>',path.read_text(),re.S):
        rows=[r for r in table_rows(table) if r]
        if not rows or rows[0][-5:]!=['Country','Artist','Song','Points','Place']:
            continue
        data=[r for r in rows[1:] if len(r)==6 and r[0].isdigit()]
        if len(data)==n:
            candidates.append({codes[clean_name(r[1])]:(int(r[5]),int(r[4])) for r in data})
    if len(candidates)!=1:
        raise ValueError(f'Expected one final results table: {path}, found {len(candidates)}')
    return candidates[0]


def parse_public_matrix(path,codes):
    tables=[t for t in re.findall(r'<table\b[^>]*>.*?</table>',path.read_text(),re.S)
        if re.search(r'<caption[^>]*>Detailed televoting results of the final',t)]
    if len(tables)!=1:
        raise ValueError(f'Expected one final public matrix: {path}')
    rows=[r for r in table_rows(tables[0]) if r]
    voters=[('WLD' if clean_name(v)=='Rest of the World' else codes[clean_name(v)]) for v in rows[1]]
    if len(set(voters))!=len(voters):
        raise ValueError('Duplicate matrix voter')
    matrix={}
    for row in rows[3:]:
        if row[0]=='Contestants':row=row[1:]
        if len(row)!=len(voters)+4:
            raise ValueError(f'Unexpected matrix row length in {path}: {len(row)}')
        recipient=codes[clean_name(row[0])]
        for voter,value in zip(voters,row[4:]):
            if voter==recipient:continue
            matrix[(voter,recipient)]=int(value) if value else 0
    return voters,matrix


def great_circle(lat1,lon1,lat2,lon2,radius=6371.0088):
    a,b=math.radians(lat1),math.radians(lat2)
    dlat=b-a;dlon=math.radians(lon2-lon1)
    h=math.sin(dlat/2)**2+math.cos(a)*math.cos(b)*math.sin(dlon/2)**2
    return 2*radius*math.asin(math.sqrt(min(1,max(0,h))))


def country_history(rows,entry,window=5,start=2003,kind='final'):
    t=entry['year'];country=entry['country']
    selected=[r for r in rows if r['country']==country and max(start,t-window)<=r['year']<t and r['year']!=2020]
    n=len(selected)
    if kind=='final':
        total=sum((r['n_finalists']-r['place_final'])/(r['n_finalists']-1) for r in selected)
        strength=(total+1.5)/(n+3)
    elif kind=='split':
        total=sum((r['place_jury']-r['place_tele'])/(r['n_finalists']-1) for r in selected)
        strength=total/(n+3)
    else:raise ValueError(kind)
    return strength,math.log1p(n),selected


def pair_history(index,voter,recipient,year,window=5,start=2016):
    selected=[r for r in index.get((voter,recipient),[]) if max(start,year-window)<=r['year']<year and r['year']!=2020]
    support=sum(r['public_points']-58/r['candidate_count'] for r in selected)/(len(selected)+3)
    return support,math.log1p(len(selected)),selected


def build(output=OUT,report_dir=REPORT,entry_file=None):
    report_dir.mkdir(parents=True,exist_ok=True)
    config=load_json(DECISIONS)
    checks=[];comparisons=[];contributions=[];inputs={DECISIONS,Path(__file__).resolve()}
    def check(name,condition,detail,year=None):
        checks.append(dict(year=year,check=name,status='pass' if condition else 'fail',detail=detail))
        if not condition:
            dump_csv(report_dir/'checks.csv',checks)
            raise ValueError(f'{name}: {detail}')
    check('fixed_specification_constants',
        (config['history_window_calendar_years'],config['smoothing_pseudocount'],
         config['history_A_source_start'],config['comparable_split_start'],config['cancelled_years'])
        ==(5,3,2003,2016,[2020]),'Constants must match STAGES_1_4_SPECIFICATION.md; changes require an explicit specification update')
    # Validate all source bytes first; existing outputs survive any failed build.
    for root,manifest_path in [(RAW,RAW/'manifest.json'),(ENTRY_RAW,ENTRY_RAW/'manifest.json')]:
        inputs.add(manifest_path)
        for record in load_json(manifest_path):
            if root==ENTRY_RAW and not (record['file'].startswith('api/') or record['file'].startswith('validation/')):
                continue
            path=root/record['file']
            check('source_hash',path.exists() and sha256(path)==record['sha256'],str(path.relative_to(ROOT)))
            inputs.add(path)
    entry_file=entry_file or OUT/'entries.csv';inputs.add(entry_file)
    entries=read_csv(entry_file)
    for e in entries:
        for field in ['year','n_finalists','place_final','place_tele','place_jury','points_final','points_tele','points_jury',
                'target_top10','target_audience_above_jury']:
            e[field]=numeric(e[field],True)
        for field in CORE_FEATURES:e[field]=numeric(e[field])
    check('base_entry_count',len(entries)==437,'437 base entries required')
    base_manifest=load_json(ROOT/'reports/entries/build_manifest.json')
    check('base_entries_hash',sha256(entry_file)==base_manifest['entries_sha256'],'Run build_entries.py after changing the base table')
    names={r['country']:r['country_name'] for r in read_csv(ROOT/'data/config/countries.csv')}
    codes={v:k for k,v in names.items()};codes.update({'Serbia and Montenegro':'CS','Serbia & Montenegro':'CS'})
    inputs.update([ROOT/'data/config/countries.csv',ROOT/'reports/entries/build_manifest.json'])
    # History-only years deliberately preserve published shared places.
    history=[]
    for year in range(2003,2008):
        directory=RAW/f'history/{year}'
        final=load_json(directory/'final.json')
        metadata={r['id']:r for p in (directory/'contestants').glob('*.json') for r in [load_json(p)]}
        n=len(final['performances'])
        check('history_expected_size',n=={2003:26,2004:24,2005:24,2006:24,2007:24}[year],str(n),year)
        validation=parse_history_validation(RAW/f'validation/Eurovision_{year}.html',n,codes)
        countries=[]
        for p in final['performances']:
            country=metadata[p['contestantId']]['country'];countries.append(country)
            points=next(s['points'] for s in p['scores'] if s['name']=='total')
            check('history_results_reconciled',validation.get(country)==(p['place'],points),country,year)
            history.append(dict(year=year,country=country,n_finalists=n,place_final=p['place'],points_final=points,
                source_file=str((directory/'final.json').relative_to(ROOT)),role='history_only'))
        check('history_unique_countries',len(countries)==len(set(countries)),str(n),year)
    history.extend(dict(year=e['year'],country=e['country'],n_finalists=e['n_finalists'],place_final=e['place_final'],
        points_final=e['points_final'],source_file=e['result_source_file'],role='model_period') for e in entries)
    check('all_A_history_years_present',{r['year'] for r in history}==set(range(2003,2026))-{2020},'2003-2025 excluding 2020')
    split_rows=[e for e in entries if e['year']>=2016]
    check('all_B_history_years_present',{e['year'] for e in split_rows}==set(VOTE_YEARS),'Comparable era starts in 2016')
    enriched=[]
    for e in entries:
        r=dict(e);year=e['year']
        strength,count,selected=country_history(history,e,start=2003)
        r.update(past_final_strength=strength,past_final_count=count,past_final_n=len(selected),
            history_window_start=year-5,history_window_end=year-1,
            past_final_source_years=';'.join(str(s['year']) for s in selected),country_history_complete=1)
        for s in selected:
            contributions.append(dict(task='A',forecast_year=year,country=e['country'],voter='',recipient='',
                source_year=s['year'],value=(s['n_finalists']-s['place_final'])/(s['n_finalists']-1),
                source_file=s['source_file']))
        if year>=2016:
            advantage,count,selected=country_history(split_rows,e,start=2016,kind='split')
            r.update(past_audience_advantage=advantage,past_split_count=count,past_split_n=len(selected),
                past_split_source_years=';'.join(str(s['year']) for s in selected),split_history_complete=1)
            for s in selected:
                contributions.append(dict(task='B',forecast_year=year,country=e['country'],voter='',recipient='',
                    source_year=s['year'],value=(s['place_jury']-s['place_tele'])/(s['n_finalists']-1),source_file=s['result_source_file']))
        else:
            r.update(past_audience_advantage=None,past_split_count=None,past_split_n=None,past_split_source_years='',split_history_complete=None)
        enriched.append(r)
    # Geography uses explicit admin-0 border records, no approximate polygon touching test.
    geo=RAW/'geography'
    places=load_json(geo/'ne_10m_populated_places.geojson')['features']
    capitals=[]
    for country in sorted(names):
        candidates=[f for f in places if f['properties']['ISO_A2']==country and f['properties']['ADM0CAP']==1]
        check('unique_capital',len(candidates)==1,country)
        f=candidates[0];p=f['properties']
        capitals.append(dict(country=country,country_name=names[country],capital=p['NAME'],
            latitude=p['LATITUDE'],longitude=p['LONGITUDE'],natural_earth_a3=p['ADM0_A3'],natural_earth_id=p['NE_ID'],
            source_version='Natural Earth v5.1.2'))
    capital_map={c['country']:c for c in capitals}
    a3_to_country={c['natural_earth_a3']:c['country'] for c in capitals}
    border_records=[];border_set=set()
    for f in load_json(geo/'ne_10m_admin_0_boundary_lines_land.geojson')['features']:
        p=f['properties'];a=a3_to_country.get(p['ADM0_A3_L']);b=a3_to_country.get(p['ADM0_A3_R'])
        if a and b and a!=b:
            included=int(p['TYPE']=='Land')
            if included:border_set.add(tuple(sorted((a,b))))
            border_records.append(dict(country_a=a,country_b=b,included=included,type=p['TYPE'],
                classification=p['FEATURECLA'],natural_earth_id=p['ne_id']))
    pairs=[]
    for a in sorted(names):
        for b in sorted(names):
            if a==b:continue
            ca,cb=capital_map[a],capital_map[b]
            distance=great_circle(ca['latitude'],ca['longitude'],cb['latitude'],cb['longitude'],config['geography']['earth_radius_km'])
            pairs.append(dict(voter=a,recipient=b,distance_km=distance,log_distance=math.log1p(distance),
                shared_border=int(tuple(sorted((a,b))) in border_set),capital_voter=ca['capital'],capital_recipient=cb['capital'],
                source_version='Natural Earth v5.1.2',valid_for_model_period='2016-2025'))
    pair_map={(r['voter'],r['recipient']):r for r in pairs}
    for r in pairs:
        reverse=pair_map[r['recipient'],r['voter']]
        check('geography_symmetric',abs(r['distance_km']-reverse['distance_km'])<1e-9 and r['shared_border']==reverse['shared_border'],f"{r['voter']}->{r['recipient']}")
    for a,b,expected in [('GB','IE',1),('IT','SM',1),('AZ','TR',1),('AM','AZ',1),('RU','UA',1),
        ('GB','ES',0),('GB','CY',0),('FR','NL',0),('DK','SE',0),('AU','NZ',0)]:
        if (a,b) in pair_map:check('border_control',pair_map[a,b]['shared_border']==expected,f'{a}-{b}={expected}')
    # Cross-check all national public points against a separate published matrix.
    votes=[];non_national=[];groups=[];rules=[];exclusions=[];matrix_comparisons=[]
    for year in VOTE_YEARS:
        final_path=ENTRY_RAW/f'api/{year}/rounds/final.json'
        final=load_json(final_path)
        meta={r['id']:r for f in (ENTRY_RAW/f'api/{year}/contestants').glob('*.json') for r in [load_json(f)]}
        country_to_performance={meta[p['contestantId']]['country']:p for p in final['performances']}
        finalists=set(country_to_performance)
        voter_list,validation=parse_public_matrix(ENTRY_RAW/f'validation/{year}.html',codes)
        api_voters=set().union(*(next(s['votes'].keys() for s in p['scores'] if s['name']=='public') for p in final['performances']))
        check('complete_voter_set',api_voters==set(voter_list),str(sorted(api_voters)),year)
        # The SM source has an explicit Aggregated televote heading in every relevant edition.
        sm_source=RAW/f'validation/San_Marino_{year}.html'
        check('SM_substitution_evidence',bool(re.search('Aggregated televote|aggregated televote',sm_source.read_text())),str(sm_source.relative_to(ROOT)),year)
        substitutes=set(config['voting']['substitute_public_voters'][str(year)])
        check('substitutes_present',substitutes<=api_voters,str(sorted(substitutes)),year)
        for voter in sorted(api_voters):
            candidates=sorted(finalists-{voter})
            group=[]
            real=voter!='WLD' and voter not in substitutes
            vote_type='rest_of_world' if voter=='WLD' else 'aggregated_substitute' if voter in substitutes else 'real_national_televote'
            for recipient in candidates:
                p=country_to_performance[recipient]
                score=next(s for s in p['scores'] if s['name']=='public')
                check('explicit_primary_vote',voter in score['votes'],f'{voter}->{recipient}',year)
                points=score['votes'][voter]
                check('public_matrix_match',validation.get((voter,recipient))==points,f'{voter}->{recipient}',year)
                matrix_comparisons.append(dict(year=year,voter=voter,recipient=recipient,api_points=points,
                    validation_points=validation[voter,recipient],status='match'))
                group.append(dict(year=year,voter=voter,recipient=recipient,candidate_count=len(candidates),
                    public_points=points,official_channel_winner=int(points==12),
                    target_first_place=int(points==12) if real else None,vote_type=vote_type,eligible_C=int(real),
                    verification_status='matrix_reconciled_exception_register_reviewed',
                    source_file=str(final_path.relative_to(ROOT)),
                    validation_file=f'data/raw/entries/validation/{year}.html',
                    type_source_file=str(sm_source.relative_to(ROOT)) if voter in substitutes else 'data/config/remaining_decisions.json'))
            check('complete_standard_scale',sorted(r['public_points'] for r in group if r['public_points']>0)==sorted(SCALE),voter,year)
            check('group_total_58',sum(r['public_points'] for r in group)==58,voter,year)
            check('one_group_winner',sum(r['official_channel_winner'] for r in group)==1,voter,year)
            check('candidate_set_complete',{r['recipient'] for r in group}==finalists-{voter},voter,year)
            groups.append(dict(year=year,voter=voter,candidate_count=len(candidates),vote_type=vote_type,
                eligible_C=int(real),winner=next(r['recipient'] for r in group if r['official_channel_winner']),
                source_file=str(final_path.relative_to(ROOT))))
            if not real:
                exclusions.append(dict(year=year,voter=voter,scope='entire_choice_group',rows=len(group),
                    reason=vote_type,source=str(sm_source.relative_to(ROOT)) if voter in substitutes else str(final_path.relative_to(ROOT))))
            (non_national if voter=='WLD' else votes).extend(group)
        rules.append(dict(year=year,final_date=final['date'],cutoff_utc=final['date']+'T12:00:00+00:00',
            finalists=len(finalists),national_voters=len(api_voters-{'WLD'}),rest_of_world=int('WLD' in api_voters),
            public_substitutes=';'.join(sorted(substitutes)),
            jury_substitutes=';'.join(config['voting']['substitute_jury_voters'][str(year)]),
            jury_aggregation='linear' if year<2018 else 'exponential',points_per_ballot=58,
            public_points_scale='12;10;8;7;6;5;4;3;2;1',
            eligible_national_groups=sum(g['eligible_C'] for g in groups if g['year']==year),
            rule_source='data/config/remaining_decisions.json'))
    # Only actual national ballots contribute to the directed pair history.
    history_index=collections.defaultdict(list)
    for row in votes:
        if row['eligible_C']:history_index[row['voter'],row['recipient']].append(row)
    check('pair_history_has_all_years',{r['year'] for rs in history_index.values() for r in rs}==set(VOTE_YEARS),'2016-2025 excluding 2020')
    enriched_map={(e['year'],e['country']):e for e in enriched}
    for row in votes:
        year=row['year'];voter=row['voter'];recipient=row['recipient']
        support,count,selected=pair_history(history_index,voter,recipient,year)
        row.update(past_pair_support=support,past_pair_count=count,past_pair_n=len(selected),
            past_pair_source_years=';'.join(str(s['year']) for s in selected),pair_history_complete=1)
        row.update({field:pair_map[voter,recipient][field] for field in ['distance_km','log_distance','shared_border']})
        row.update({field:enriched_map[year,recipient][field] for field in CORE_FEATURES})
        for s in selected:
            contributions.append(dict(task='C',forecast_year=year,country='',voter=voter,recipient=recipient,
                source_year=s['year'],value=s['public_points']-58/s['candidate_count'],source_file=s['source_file']))
    for c in contributions:
        check('strict_five_year_history',c['forecast_year']-5<=c['source_year']<c['forecast_year'] and c['source_year']!=2020,
            f"{c['task']}:{c['source_year']}->{c['forecast_year']}")
    # Model tables expose explicit predictor allowlists. No full-dataset scaling/imputation.
    model_A=[{k:e[k] for k in ['year','country','n_finalists','place_final','target_top10']+FEATURES_A} for e in enriched]
    model_B=[{k:e[k] for k in ['year','country','target_audience_above_jury']+FEATURES_B} for e in enriched if e['year']>=2016]
    model_C=[{k:r[k] for k in ['year','voter','recipient','candidate_count','target_first_place']+FEATURES_C} for r in votes if r['eligible_C']]
    for task,rows,fields in [('A',model_A,FEATURES_A),('B',model_B,FEATURES_B),('C',model_C,FEATURES_C)]:
        check('feature_dimension_'+task,len(fields)=={'A':11,'B':11,'C':13}[task],str(fields))
        check('complete_model_features_'+task,all(all(r[f] is not None and math.isfinite(r[f]) for f in fields) for r in rows),str(len(rows)))
    for key,group in __import__('itertools').groupby(model_C,key=lambda r:(r['year'],r['voter'])):
        rows=list(group)
        check('model_C_full_choice_set',len(rows)==rows[0]['candidate_count'] and sum(r['target_first_place'] for r in rows)==1,str(key))
    coverage=[]
    for year in VOTE_YEARS:
        rows=[r for r in votes if r['year']==year]
        valid=[r for r in rows if r['eligible_C']]
        coverage.append(dict(year=year,finalists=sum(e['year']==year for e in entries),
            national_groups=sum(g['year']==year and g['voter']!='WLD' for g in groups),
            real_groups=sum(g['year']==year and g['eligible_C'] for g in groups),
            excluded_substitute_groups=sum(g['year']==year and g['vote_type']=='aggregated_substitute' for g in groups),
            national_rows=len(rows),model_C_rows=len(valid),
            zero_point_real_observations=sum(r['public_points']==0 for r in valid),
            model_A_rows=sum(r['year']==year for r in model_A),model_B_rows=sum(r['year']==year for r in model_B),
            missing_core_predictors=0))
    outputs={'entries_features.csv':enriched,'history_results.csv':history,'country_capitals.csv':capitals,
        'country_pairs.csv':pairs,'national_votes.csv':votes,'non_national_votes.csv':non_national,
        'contest_rules.csv':rules,'model_A.csv':model_A,'model_B.csv':model_B,'model_C.csv':model_C}
    # All validation precedes writes to final tables.
    for name,rows in outputs.items():dump_csv(output/name,rows)
    for name,rows in {'checks.csv':checks,'coverage.csv':coverage,'exclusions.csv':exclusions,
        'choice_groups.csv':groups,'history_contributions.csv':contributions,'border_records.csv':border_records,
        'national_source_comparisons.csv':matrix_comparisons}.items():dump_csv(report_dir/name,rows)
    feature_sets=dict(A=FEATURES_A,B=FEATURES_B,C=FEATURES_C,
        preprocessing='Raw features. Fit standardization and any imputation separately on each training fold.',
        existing_limitations='Published Song Science features accepted by user; dated pre-final running-order publications checked. Market supplement is added by build_final_inputs.py.')
    (output/'feature_sets.json').write_text(json.dumps(feature_sets,indent=2)+'\n')
    inputs.update(RAW.glob('voting_evidence/*'))
    manifest=dict(output_rows={name:len(rows) for name,rows in outputs.items()},checks_passed=len(checks),
        valid_choice_groups=sum(g['eligible_C'] for g in groups),
        input_fingerprints=[dict(file=str(p.relative_to(ROOT)),sha256=sha256(p)) for p in sorted(inputs)],
        output_fingerprints=[dict(file=str((output/name).relative_to(ROOT)) if (output/name).is_relative_to(ROOT) else name,
            sha256=sha256(output/name)) for name in list(outputs)+['feature_sets.json']])
    (report_dir/'build_manifest.json').write_text(json.dumps(manifest,indent=2)+'\n')
    write_report(report_dir,outputs,coverage,checks,groups,border_set)
    print(json.dumps(dict(rows=manifest['output_rows'],checks=len(checks),valid_choice_groups=manifest['valid_choice_groups']),indent=2))
    return outputs


def write_report(directory,outputs,coverage,checks,groups,border_set):
    lines=['# История стран, национальные голосования и география','',
        'Запрошенные блоки собраны. Основная таблица `entries.csv` сохранена как исходная проверенная база; '
        'исторические столбцы находятся в `entries_features.csv`. Все конечные таблицы лежат в `data/processed/`.','',
        '## Что для какой модели','',
        '| Таблица | Наблюдений | Назначение |', '|---|---:|---|',
        f'| model_A.csv | {len(outputs["model_A.csv"])} | Топ-10, Plackett–Luce M1/M2: 9 общих + 2 исторических признака страны |',
        f'| model_B.csv | {len(outputs["model_B.csv"])} | Probit M1/M2: 9 общих + 2 исторических признака раздельных мест |',
        f'| model_C.csv | {len(outputs["model_C.csv"])} | Multinomial logit M1/M2: 9 общих + 2 признака пары + расстояние и граница |','',
        'M1 и M2 используют одинаковые столбцы. Эффекты стран и пар для M2 будут оцениваться при обучении. '
        'Точный список предикторов — `feature_sets.json`; идентификаторы, цели и сырые баллы в него не входят.','',
        '## Исторические признаки','',
        '- Окно — предыдущие пять календарных лет, 2020 пропускается без расширения окна. '
        'Для каждой строки источники строго раньше прогнозируемого года.',
        '- A: `(Σq + 1.5)/(n+3)`, `q=(N-R)/(N-1)`, и `log(1+n)`. '
        'Предыстория 2003–2007 догружена, все её места и баллы сверены с отдельными опубликованными таблицами.',
        '- В 2003–2007 сохранены опубликованные совместные места. Эти годы не используются как рейтинги для likelihood Plackett–Luce.',
        '- B: `Σ[(R_jury-R_tele)/(N-1)]/(n+3)` и `log(1+n)`. Сопоставимая история начинается в 2016, '
        'поэтому в 2016 значения равны нейтральному prior и нулевому счётчику.',
        '- C: `Σ[points_AB - 58/K_As]/(n+3)` и `log(1+n)`. Ноль баллов — наблюдение; '
        'отсутствующий финалист, самоголосование, заменённое голосование и RoW в знаменатель не входят.',
        '- CS (Сербия и Черногория) не переносится на историю RS или ME. '
        'Каждый вклад в исторический признак можно проследить в `history_contributions.csv`.','',
        '## Национальные голосования','',
        '| Год | Национальные группы | Настоящие группы для C | Исключённые замены | Строк C |',
        '|---|---:|---:|---:|---:|']
    for c in coverage:lines.append(f'| {c["year"]} | {c["national_groups"]} | {c["real_groups"]} | {c["excluded_substitute_groups"]} | {c["model_C_rows"]} |')
    lines += ['',
        '- Каждая группа содержит полный набор допустимых получателей. Сохранены голосующие страны, не прошедшие в финал. '
        'Все пары в первичном JSON должны присутствовать явно, включая нули.',
        '- Каждый национальный public-балл сверён с отдельной опубликованной матрицей. Проверены сумма 58, стандартная шкала и единственный победитель.',
        '- Сан-Марино: исключены целые группы всех девяти лет. '
        'Систему замен описывает [NDR](https://www.eurovision.de/news/Fragen-und-Antworten,faq117.html); '
        'в сохранённых годовых страницах Сан-Марино каждый раз есть явная маркировка Aggregated televote. '
        'В `national_votes.csv` эти группы сохранены с `eligible_C=0`, целевая метка пуста. Они не участвуют в истории пар.',
        '- Rest of the World вынесен в `non_national_votes.csv`, не используется как страна A. '
        'Его вклад в общий результат участника остаётся в исходной entries.',
        '- Замены жюри 2019/2022 отражены в `contest_rules.csv`. В 2022 не исключались настоящие public-голоса стран '
        'только из-за замены их жюри. Правила замен public и jury разделены.',
        '- Классификация «настоящее» опирается на опубликованный официальный public-результат и проверенный реестр исключений. '
        'Это проверка источников, а не аудит сырых телефонных голосов. Неполная матрица или неизвестное исключение останавливает сборку.','',
        '## География','',
        f'- Столицы: {len(outputs["country_capitals.csv"])} стран; направленные пары: {len(outputs["country_pairs.csv"])}. '
        f'Общую границу имеют {len(border_set)} неупорядоченных пар.',
        '- Natural Earth v5.1.2, масштаб 1:10m. Для каждой страны ровно одна столица ADM0CAP=1. '
        'Координаты и исходный NE_ID сохранены. Использованы Amsterdam, Podgorica и Jerusalem согласно справочнику.',
        '- Расстояние — по большой окружности, радиус Земли 6371.0088 км; `log_distance=log(1+distance_km)`.',
        '- Граница берётся из явных записей TYPE=Land между кодами admin-0. Морские индикаторы исключены. '
        'Отдельно закодированные зависимые территории не присоединяются к стране: GB–ES, GB–CY и FR–NL равны нулю; GB–IE, IT–SM и AZ–TR равны единице.',
        '- География статична в пределах C (2016–2025): изменения линий в RU–UA, RU–GE, AM–AZ не меняют наличие сухопутного соседства. '
        'Выбор кодов территорий и спорных случаев зафиксирован до обучения в `remaining_decisions.json`. '
        'Население городов и другие меняющиеся атрибуты карты не использованы.','',
        '## Проверки и воспроизведение','',
        f'Все {len(checks)} проверок прошли. Новые таблицы не содержат пропусков в 11/11/13 предикторах. '
        'Стандартизация и импутация на всей выборке не выполнялись.',
        'Сборка без сети и дополнительных пакетов: `python3 scripts/build_remaining.py`. '
        'Полный запуск: `python3 scripts/prepare_data.py`. Тесты: `python3 -m unittest discover -s tests -v`.',
        'Хеши входов/выходов — `build_manifest.json`; национальная сверка — `national_source_comparisons.csv`; '
        'полные исключённые группы — `exclusions.csv`; состав каждой группы — `choice_groups.csv`; '
        'исходные граничные линии — `border_records.csv`.','',
        '## Граница выполненного','',
        'История стран, национальные голосования, история пар, расстояния и границы реализованы. '
        'Готовые признаки песен приняты по решению пользователя; состав и порядок проверены по датированным публикациям. '
        'Рыночное дополнение и его ограничения описаны в `reports/final_inputs/README.md`. Модели не обучались.','']
    (directory/'README.md').write_text('\n'.join(lines),encoding='utf-8')


if __name__=='__main__':
    parser=argparse.ArgumentParser(description=__doc__)
    parser.add_argument('--output-dir',type=Path,default=OUT)
    parser.add_argument('--report-dir',type=Path,default=REPORT)
    args=parser.parse_args();build(args.output_dir,args.report_dir)


### 3.3. Котировки Eurovisionworld

Парсим сохранённые строки таблиц закрытого рынка. Во всех годах используем BET365, BETFRED, BoyleSports, SkyBet и Unibet. Сначала нормируем `1/odds` внутри каждого букмекера по финалистам, затем усредняем пять вероятностей. `market_strength = log(max(p_market, 1e-6))`. Время измерения остаётся неизвестным.

In [ ]:
%%writefile {BUILD_DIR}/scripts/market_archive.py
"""One primary provider and a fixed five-bookmaker panel, offline and hash-locked."""
import collections
import json
import math
import re
from pathlib import Path
from build_entries import ROOT, read_csv, load_json, sha256

YEARS=[2015,2016,2017,2018,2019,2021,2022,2023,2024,2025]
PANEL=['BET365','BETFRED','BOYLESPORTS','SKYBET','UNIBET']

def read_archive_tables():
    locks=load_json(ROOT/'data/config/market_archive_sources.json')
    pages=collections.defaultdict(dict)
    for item in locks:
        path=ROOT/item['file']
        if sha256(path)!=item['sha256']:
            raise ValueError(f'Market archive hash changed: {path}')
        for section in re.split(r'\n(?=Odds Eurovision Song Contest \d{4} \()',path.read_text()):
            match=re.search(r'Odds Eurovision Song Contest (\d{4})',section)
            if not match:
                continue
            year=int(match[1])
            for number,line in re.findall(r'^L(\d+): (.*)$',section,re.M):
                number=int(number)
                if number in pages[year] and pages[year][number]!=line:
                    raise ValueError(f'Archive page changed between captures: {year} line {number}')
                pages[year][number]=line
    names={r['country_name']:r['country'] for r in read_csv(ROOT/'data/config/countries.csv')}
    names.update({'Czech Republic':'CZ','The Netherlands':'NL','Macedonia':'MK',
        'F.Y.R. Macedonia':'MK','Bosnia and Herzegovina':'BA','Turkey':'TR','Türkiye':'TR'})
    result=[]
    for year in YEARS:
        lines=[line for _,line in sorted(pages[year].items())]
        if 'Closed event' not in lines or 'The odds will not be updated' not in lines:
            raise ValueError(f'Not a closed archive: {year}')
        row_indexes=[i for i,line in enumerate(lines) if re.match(r'^\d{1,2}\s+\|',line) and '†' in line]
        if not row_indexes:
            raise ValueError(f'No archive table rows: {year}')
        start=max(i for i,line in enumerate(lines[:row_indexes[0]]) if re.fullmatch(r'[\s|:-]+',line) and '|' in line)+1
        header=' '.join(lines[start:row_indexes[0]])
        books=[re.sub(r'[^A-Z0-9]','',x.upper()) for x in header.split('|')]
        books=[b for b in books if b and b not in {'WINNINGCHANCE','HISTORICALODDS'}]
        for index in row_indexes:
            cells=[x.strip() for x in lines[index].split('|')]
            found=[]
            for j,cell in enumerate(cells):
                for label in re.findall(r'【\d+†([^】]+)】',cell):
                    label=label.replace('UKUnited Kingdom','United Kingdom').replace('UKUnitedKingdom','United Kingdom')
                    label=label.replace('N.MacedoniaNorth Macedonia','North Macedonia')
                    matches=[name for name in names if label.startswith(name)]
                    if matches:
                        country=names[max(matches,key=len)]
                        found.append((j,country))
            if len(found)!=1:
                raise ValueError(f'Unresolved country {year}: {cells}')
            j,country=found[0]
            odds=cells[j+1:]
            if odds and '%' in odds[0]:
                odds=odds[1:]
            if len(odds)!=len(books):
                raise ValueError(f'Archive columns {year} {country}: {len(odds)} prices, {len(books)} books; {books}')
            for bookmaker,value in zip(books,odds):
                if bookmaker not in PANEL:
                    continue
                try:
                    price=float(value)
                except ValueError:
                    raise ValueError(f'Missing fixed-panel quote: {year} {country} {bookmaker}: {value!r}')
                if not math.isfinite(price) or price<=1:
                    raise ValueError(f'Invalid decimal odds: {year} {country} {bookmaker}')
                result.append(dict(year=year,country=country,bookmaker=bookmaker,decimal_odds=price,
                    source_url=f'https://eurovisionworld.com/odds/eurovision-{year}',
                    source_line=next(k for k,v in pages[year].items() if v==lines[index]),
                    source_status='closed_event_archive_timestamp_unknown'))
    keys=[(r['year'],r['country'],r['bookmaker']) for r in result]
    if len(keys)!=len(set(keys)):
        raise ValueError('Duplicate archive quote')
    return result

def archive_market(entries):
    quotes=read_archive_tables()
    features=[]
    for year in YEARS:
        countries={r['country'] for r in entries if int(r['year'])==year}
        prices={b:{} for b in PANEL}
        for row in quotes:
            if row['year']==year:
                prices[row['bookmaker']][row['country']]=row['decimal_odds']
        for book,table in prices.items():
            if set(table)!=countries:
                raise ValueError(f'Incomplete finalist panel: {year} {book}: missing {countries-set(table)}, extra {set(table)-countries}')
        probabilities={}
        for book,table in prices.items():
            total=sum(1/value for value in table.values())
            probabilities[book]={country:(1/value)/total for country,value in table.items()}
        for country in sorted(countries):
            p=sum(probabilities[book][country] for book in PANEL)/len(PANEL)
            features.append(dict(year=year,country=country,p_market=p,
                market_strength=math.log(max(p,1e-6)),market='win',provider='Eurovisionworld',
                bookmaker_count=len(PANEL),bookmaker_panel=';'.join(PANEL),
                captured_at_source='',captured_at_utc='',timestamp_status='unknown_accepted_by_user',
                source_status='closed_event_archive',
                cutoff_verified=0,
                source_url=f'https://eurovisionworld.com/odds/eurovision-{year}',
                normalization='For each of the same five books normalize inverse decimal odds over eligible finalists, then average probabilities.'))
    return features,quotes


### 3.4. Итоговые входы моделей

Проверяем 437 опубликованных слотов и присоединяем рынок по `(year, country)`; для C — по получателю. Отдельно сохраняем Core на тех же наблюдениях, чтобы сравнение с рынком было корректным. Функция `market_snapshot` сохранена для старых тестов альтернативного источника; текущая сборка вызывает только `archive_market` из предыдущей ячейки.

In [ ]:
%%writefile {BUILD_DIR}/scripts/build_final_inputs.py
#!/usr/bin/env python3
"""Offline final input audit and the user-accepted archival market experiment."""
import datetime as dt
import json
import math
from pathlib import Path
from build_entries import ROOT, OUT, read_csv, dump_csv, load_json, sha256

RAW=ROOT/'data/raw/final_validation'
REPORT=ROOT/'reports/final_inputs'

def validate_running_orders(entries):
    evidence=load_json(ROOT/'data/config/running_order_evidence.json')
    expected={(int(r['year']),r['country']):int(r['running_slot']) for r in entries}
    found={}
    sources=[]
    for source in evidence:
        path=ROOT/source['source_file']
        if sha256(path)!=source['source_sha256']:
            raise ValueError(f'Running-order source changed: {path}')
        year=source['year']
        final_date=next(r['final_date'] for r in entries if int(r['year'])==year)
        # A dated prior-day publication predates the next day's noon UTC,
        # including date-only sources in any civil timezone (-12..+14).
        published=source['published_at_source']
        if published[:10]>=final_date:
            raise ValueError(f'Publication is not before final day: {year}')
        for slot in source['slots']:
            key=(year,slot['country'])
            if key in found:
                raise ValueError(f'Duplicate published slot {key}')
            found[key]=slot['running_slot']
            sources.append(dict(year=year,country=slot['country'],running_slot=slot['running_slot'],
                published_at_source=published,source_url=source['source_url'],
                source_file=source['source_file'],note=source['note']))
    if expected!=found:
        raise ValueError('Published running order differs from the full entries table')
    return sources

def market_snapshot(year, entries):
    source=RAW/f'odds{year}_api.json'
    locked=next(r for r in json.loads((ROOT/'data/config/market_sources.json').read_text()) if r['file']==source.name)
    if sha256(source)!=locked['sha256']:
        raise ValueError(f'Market source hash changed: {source}')
    data=load_json(source)
    group=[r for r in entries if int(r['year'])==year]
    final_date=group[0]['final_date']
    selected=final_date+' 00:00'
    # The API omits its timezone. Do not manufacture an exact UTC timestamp:
    # midnight is <= noon UTC in every civil timezone, including UTC-12.
    clock=dt.datetime.fromisoformat(selected).replace(tzinfo=dt.timezone.utc)
    lower=clock-dt.timedelta(hours=14)
    upper=clock+dt.timedelta(hours=12)
    cutoff=dt.datetime.fromisoformat(group[0]['cutoff_utc'])
    if upper>cutoff or (cutoff-lower)>dt.timedelta(hours=26):
        raise ValueError('Market timestamp bound violates pre-cutoff policy')
    values={}
    for entry in group:
        observations=[p for p in data[entry['country']] if p['x']==selected]
        if len(observations)!=1:
            raise ValueError(f'No unique simultaneous midnight quote: {year} {entry["country"]}')
        value=float(observations[0]['y'])/100
        if not math.isfinite(value) or not 0<value<=1:
            raise ValueError('Invalid market probability')
        values[entry['country']]=value
    denominator=sum(values.values())
    return [dict(year=year,country=country,market='win',provider='EurovisionOdds.eu',
        source_probability=value,p_market=value/denominator,
        market_strength=math.log(max(value/denominator,1e-6)),
        captured_at_source=selected,captured_at_utc='',
        captured_at_utc_lower=lower.isoformat(),captured_at_utc_upper=upper.isoformat(),
        timestamp_status='timezone_unknown_bounded_all_civil_offsets',
        max_age_hours=(cutoff-lower).total_seconds()/3600,
        source_url=f'https://www.eurovisionodds.eu/get_odds.php?year={year}',
        source_file=str(source.relative_to(ROOT)),source_sha256=sha256(source),
        normalization='source probabilities conditioned on eligible finalists; not a bookmaker-margin reconstruction')
        for country,value in sorted(values.items())]

def build(output=OUT, report_dir=REPORT):
    from market_archive import archive_market, YEARS, PANEL
    entries=read_csv(output/'entries.csv')
    running=validate_running_orders(entries)
    market,quotes=archive_market(entries)
    lookup={(str(r['year']),r['country']):r for r in market}
    plans={}
    for task in ['A','B','C']:
        model=read_csv(output/f'model_{task}.csv')
        country_field='recipient' if task=='C' else 'country'
        restricted=[r for r in model if (r['year'],r[country_field]) in lookup]
        augmented=[]
        for r in restricted:
            m=lookup[(r['year'],r[country_field])]
            augmented.append(dict(r,p_market=m['p_market'],market_strength=m['market_strength'],
                bookmaker_count=m['bookmaker_count'],market_provider=m['provider'],
                market_timestamp_status=m['timestamp_status'],market_cutoff_verified=m['cutoff_verified']))
        dump_csv(output/f'model_{task}_core_market_years.csv',restricted)
        dump_csv(output/f'model_{task}_market.csv',augmented)
        initial=[2015,2016,2017,2018,2019] if task=='A' else [2016,2017,2018,2019]
        plans[task]=dict(rows=len(augmented),train_initial=initial,test=[2021,2022,2023,2024,2025],
            rule='Expanding train: only earlier market years. Core comparison uses exactly the same rows and folds.')
    dump_csv(output/'running_order_sources.csv',running)
    dump_csv(output/'odds_snapshots.csv',market)
    dump_csv(output/'bookmaker_odds.csv',quotes)
    dump_csv(output/'market_features.csv',[{k:r[k] for k in ['year','country','p_market','market_strength']} for r in market])
    feature_sets=load_json(output/'feature_sets.json')
    for task in ['A','B','C']:
        feature_sets[task+'_market']=feature_sets[task]+['market_strength']
    feature_sets['existing_limitations']='Published Song Science features accepted by user. Market uses one Eurovisionworld closed-event archive and the same five bookmakers in 2015–2025 excluding 2020. Exact snapshot time and pre-cutoff availability are unverified and explicitly accepted as an assumption.'
    (output/'feature_sets.json').write_text(json.dumps(feature_sets,indent=2)+'\n')
    report_dir.mkdir(parents=True,exist_ok=True)
    summary=dict(core_training_ready=len(entries),published_running_slots=len(running),market_finalists=len(market),
        market_years=YEARS,market_plans=plans,bookmaker_panel=PANEL,raw_quotes=len(quotes),
        audio_policy='Use published GitHub features as requested; authors declare studio recordings. Independent per-track verification waived, not claimed completed.',
        market_policy='User accepts unknown measurement time. One primary source and closed-event archive format throughout; same bookmaker panel and normalization. Equal clock time, common time-to-final and availability by 12:00 UTC are not verified. Interpret as an archival-odds experiment, not a timestamp-verified pre-final backtest.',
        provider_method='For each of the same five bookmakers normalize inverse decimal odds over the full finalist set, then average the five probability vectors.',
        inputs=[dict(file=str(p.relative_to(ROOT)),sha256=sha256(p)) for p in [ROOT/'data/config/running_order_evidence.json',ROOT/'data/config/market_archive_sources.json',ROOT/'scripts/market_archive.py',*[ROOT/r['file'] for r in load_json(ROOT/'data/config/market_archive_sources.json')],Path(__file__).resolve()]],
        checks=dict(running_slots_exact_match=len(running),market_complete_rows=len(market),market_tasks=plans))
    (report_dir/'build_manifest.json').write_text(json.dumps(summary,ensure_ascii=False,indent=2)+'\n')
    coverage=[]
    for task in ['A','B','C','A_market','B_market','C_market']:
        model=read_csv(output/f'model_{task}.csv')
        for field in feature_sets[task]:
            observed=[float(r[field]) for r in model if r[field]!='']
            coverage.append(dict(task=task,feature=field,rows=len(model),present=len(observed),
                missing=len(model)-len(observed),years=';'.join(str(y) for y in sorted({int(r['year']) for r in model if r[field]!=''})),
                min=min(observed),max=max(observed)))
    dump_csv(report_dir/'feature_coverage.csv',coverage)
    (report_dir/'README.md').write_text('''# Финальные входные данные

Core готов к реализации моделей: 437 финалистов A, 232 строки B, 8770 строк / 349 национальных голосований C.

## Порядок выступлений

437 исходных номеров за все 17 конкурсов 2008–2025 совпали с полными списками в датированных предфинальных публикациях. Источники, даты, номера и оговорка 2024: `data/processed/running_order_sources.csv`. Сохранены HTML и ответы веб-поиска; их хеши проверяются при каждой сборке. Это архив текущих копий датированных публикаций, не Wayback-снимки состояния страниц в прошлом. В 2024 исключение NL известно до cutoff; прочие номера сохранены.

## Признаки песни

По решению пользователя принимаем готовые признаки Song Science и заявленную авторами студийную основу. Значения проверены на заполненность, диапазоны и корректность соединения. Индивидуальную проверку каждого перезалива не требуем и не называем выполненной. Сырые YouTube-метаданные из предварительного исследования не являются признаками модели. `audio_url2` за 2025 хранится отдельно; готовые численные признаки не пересчитывались.

## Рынок

Обновление 25 октября 2026: по прямому решению пользователя отсутствие timestamp больше не блокирует использование архивных котировок. Использован единый первоисточник **Eurovisionworld**, архивы закрытого рынка общей победы за **2015–2025**, без отменённого 2020. Все десять лет заново прочитаны из одного архива; старый CSV MIRoVision не смешивается с новыми страницами, а временные ряды EurovisionOdds.eu не подмешиваются.

**259 финалистов, 1295 котировок, без пропусков.** Постоянная панель во всех годах: BET365, BETFRED, BoyleSports, SkyBet, Unibet. Для каждого букмекера сначала считаем `(1/odds) / sum(1/odds)` по полному допустимому набору финалистов, затем усредняем пять векторов. `market_strength = log(max(p_market, 1e-6))`. Это наш единый расчёт, а не округлённая колонка winning chance сайта. В 2024 NL исключён из допустимых получателей; состав сопоставлен с ранее проверенной таблицей финала.

**Время измерения неизвестно.** Одинаковый первоисточник и постоянная панель не доказывают одинаковое время суток, одинаковое число часов до финала или доступность котировок к основному cutoff 12:00 UTC. Такие гарантии не заявляются; в данных `timestamp_status=unknown_accepted_by_user`, `cutoff_verified=0`, UTC и исходный timestamp пусты. Это архивный рыночный эксперимент с принятым допущением о времени, а не backtest с доказанной предфинальностью рынка. Основной Core и его cutoff не изменены.

Файлы: `bookmaker_odds.csv` — все 1295 исходных цен; `odds_snapshots.csv` — 259 рассчитанных вероятностей и происхождение; `market_features.csv` — признаки; `model_*_market.csv` — готовые модельные таблицы. M1/M2 получают 12/12/14 объясняющих столбцов из `feature_sets.json`; дополнительные поля источника и статуса не являются признаками.

Размеры с рынком: A — 259 строк (2015–2025), B — 232 строки (2016–2025), C — 8770 строк / 349 полных голосований (2016–2025). Core на точно тех же наблюдениях сохранён в `*_core_market_years.csv`. Для сравнения с рынком: начальный train A — 2015–2019, B/C — 2016–2019; общие тестовые годы — 2021–2025 с расширением train. Core и Market в каждом сравнении имеют одинаковые train/test. Основной Core A по всей истории остаётся отдельным экспериментом.

Покрытие каждого признака, фактические диапазоны значений и пропуски: `reports/final_inputs/feature_coverage.csv`. Исходные страницы, строки таблиц и SHA-256 сохранены. Прежний временной набор 2023–2025 сохранён в `data/raw/market_transition/odds_snapshots_timed_2023_2025.csv` только для истории, в новый рынок не входит.

## Воспроизведение

`python3 scripts/prepare_data.py` — полностью офлайн. `python3 -m unittest discover -s tests -v` — проверки. Модели пока не обучались.
''',encoding='utf-8')
    print(f'Final inputs: {len(running)} published slots matched; {len(market)} market rows, {len(quotes)} quotes (2015–2025).')
    return summary

if __name__=='__main__':
    build()


## 4. Полная сборка A/B/C

Порядок: базовые записи → история/голоса/география → рыночное дополнение.
Обрабатываются исходники; готовых таблиц перед этим шагом в архиве нет.
При нарушении контрольных связей или хешей выполнение останавливается.

In [ ]:
sys.path.insert(0, str(BUILD_DIR / "scripts"))
# Run all можно выполнять повторно в том же ядре.
for module_name in ("build_entries", "build_remaining", "market_archive", "build_final_inputs"):
    sys.modules.pop(module_name, None)
importlib.invalidate_caches()
import build_entries
import build_remaining
import build_final_inputs

build_entries.build()
build_remaining.build()
build_final_inputs.build()
PROCESSED = BUILD_DIR / "data/processed"

## 5. Проверка воспроизведения и покрытия

Сверяем SHA-256 **всех 21 итоговых CSV** с таблицами, использованными в проекте.
Каталог содержит только ожидаемые хеши, а не готовые CSV. Проверяем размер,
пропуски в разрешённых признаках, нормировку рынка и полные голосования C.

In [ ]:
csv_checks = pd.DataFrame([
    dict(file=name, matches_original=file_sha256(PROCESSED / name) == expected)
    for name, expected in manifest["expected_csv_sha256"].items()
])
display(csv_checks)
assert csv_checks.matches_original.all(), "Таблицы отличаются от исходных экспериментов"

feature_sets = json.loads((PROCESSED / "feature_sets.json").read_text())
sizes = {"A": 437, "B": 232, "C": 8770, "A_market": 259, "B_market": 232, "C_market": 8770}
tables = {task: pd.read_csv(PROCESSED / f"model_{task}.csv") for task in sizes}
coverage = []
for task, frame in tables.items():
    assert len(frame) == sizes[task], task
    assert not frame.year.eq(2020).any()
    assert np.isfinite(frame[feature_sets[task]].to_numpy(float)).all(), task
    coverage.append(dict(task=task, rows=len(frame), first_year=int(frame.year.min()),
                         last_year=int(frame.year.max()), predictors=len(feature_sets[task]),
                         missing_predictors=int(frame[feature_sets[task]].isna().sum().sum())))
display(pd.DataFrame(coverage))
odds = pd.read_csv(PROCESSED / "odds_snapshots.csv")
assert np.allclose(odds.groupby("year").p_market.sum(), 1)
assert odds.cutoff_verified.eq(0).all()
assert len(pd.read_csv(PROCESSED / "bookmaker_odds.csv")) == 1295
for task in ["C", "C_market"]:
    c = tables[task]
    groups = c.groupby(["year", "voter"])
    assert groups.ngroups == 349
    assert groups.target_first_place.sum().eq(1).all()
    assert groups.size().eq(groups.candidate_count.first()).all()
    assert not c.voter.eq(c.recipient).any()
    assert not c.voter.isin(["SM", "WLD", "ROW"]).any()
display(tables["A_market"].head(5))

## 6. Поведенческие проверки

Оригинальные тесты проверяют ничьи, исключение NL-2024, строго прошлую историю,
отличие пропуска от нуля, направленность пар, географию и целые наборы кандидатов.
Они включают также тесты старого временного источника котировок; этот источник
не входит в текущие модельные таблицы. Внешние библиотеки для тестов не нужны.

In [ ]:
(BUILD_DIR / "tests").mkdir(exist_ok=True)

In [ ]:
%%writefile {BUILD_DIR}/tests/test_entries.py
"""Acceptance checks for the data contract and ranking edge cases. Python stdlib only."""
import contextlib
import csv
import hashlib
import importlib.util
import io
import json
import tempfile
import unittest
from pathlib import Path
from unittest.mock import patch

ROOT = Path(__file__).resolve().parents[1]
spec = importlib.util.spec_from_file_location('build_entries', ROOT/'scripts/build_entries.py')
builder = importlib.util.module_from_spec(spec)
spec.loader.exec_module(builder)


def performance(identifier, running, votes):
    return dict(contestantId=identifier, running=running,
        scores=[dict(name='public', points=sum(votes.values()), votes=votes)])


class RankingTests(unittest.TestCase):
    def test_points_precede_other_rules(self):
        rows=[performance(0,1,{'A':12}), performance(1,2,{'A':8,'B':5})]
        self.assertEqual(builder.split_ranks(rows,'public'),{1:1,0:2})

    def test_support_count_precedes_twelve_count(self):
        rows=[performance(0,1,{'A':12}), performance(1,2,{'A':8,'B':4})]
        self.assertEqual(builder.split_ranks(rows,'public'),{1:1,0:2})

    def test_twelve_point_count_precedes_ten(self):
        rows=[performance(0,2,{'A':12,'B':4}),performance(1,1,{'A':10,'B':6})]
        self.assertEqual(builder.split_ranks(rows,'public'),{0:1,1:2})

    def test_countback_goes_below_twelve(self):
        rows=[performance(0,1,{'A':7,'B':5}),performance(1,2,{'A':8,'B':4})]
        self.assertEqual(builder.split_ranks(rows,'public'),{1:1,0:2})

    def test_zero_points_use_running_order_not_id(self):
        rows=[performance(1,23,{'A':0}),performance(0,9,{'B':0})]
        self.assertEqual(builder.split_ranks(rows,'public'),{0:1,1:2})

    def test_unresolved_tie_fails(self):
        rows=[performance(0,1,{'A':12}),performance(1,1,{'B':12})]
        with self.assertRaisesRegex(ValueError,'Unresolved'):
            builder.split_ranks(rows,'public')

    def test_missing_and_zero_are_distinct(self):
        self.assertIsNone(builder.numeric(''))
        self.assertEqual(builder.numeric('0',True),0)
        with self.assertRaises(ValueError):
            builder.numeric('inf')
        with self.assertRaises(ValueError):
            builder.numeric('1.5',True)


class BuildTests(unittest.TestCase):
    @classmethod
    def setUpClass(cls):
        cls.tmp=tempfile.TemporaryDirectory()
        cls.output=Path(cls.tmp.name)/'processed'
        cls.report=Path(cls.tmp.name)/'reports'
        with contextlib.redirect_stdout(io.StringIO()):
            cls.entries=builder.build(cls.output,cls.report)
        cls.rows={(r['year'],r['country']):r for r in cls.entries}

    @classmethod
    def tearDownClass(cls):
        cls.tmp.cleanup()

    def test_complete_population_and_target_periods(self):
        self.assertEqual(len(self.entries),437)
        self.assertEqual(sum(r['target_audience_above_jury'] is not None for r in self.entries),232)
        self.assertNotIn(2020,{r['year'] for r in self.entries})
        self.assertNotIn((2024,'NL'),self.rows)
        for year in builder.YEARS:
            group=[r for r in self.entries if r['year']==year]
            self.assertEqual(sum(r['target_top10'] for r in group),10)

    def test_2024_slot_and_dense_position(self):
        row=self.rows[2024,'AT']
        self.assertEqual((row['running_slot'],row['running_order'],row['running_position']),(26,25,1.0))
        row=self.rows[2024,'IL']
        self.assertEqual((row['running_slot'],row['running_order']),(6,5))

    def test_host_is_actual_location(self):
        self.assertEqual(self.rows[2023,'GB']['host'],1)
        self.assertEqual(self.rows[2023,'UA']['host'],0)

    def test_2019_corrected_results(self):
        self.assertEqual(self.rows[2019,'SE']['points_final'],334)
        self.assertEqual(self.rows[2019,'SE']['place_final'],5)
        self.assertEqual(self.rows[2019,'MK']['place_jury'],1)

    def test_split_ranks_with_zero_and_equal_points(self):
        self.assertEqual([self.rows[2021,c]['place_tele'] for c in ['GB','ES','DE','NL']],[23,24,25,26])
        self.assertEqual(self.rows[2025,'GB']['place_jury'],10)
        self.assertEqual(self.rows[2025,'FI']['place_jury'],11)
        self.assertEqual(self.rows[2025,'CH']['points_tele'],0)

    def test_equal_split_places_give_zero(self):
        equal=[r for r in self.entries if r['place_jury'] is not None and r['place_jury']==r['place_tele']]
        self.assertTrue(equal)
        self.assertTrue(all(r['target_audience_above_jury']==0 for r in equal))

    def test_no_future_metrics_or_ids_in_table(self):
        columns=set(self.entries[0])
        self.assertFalse(columns & {'idx','contestantId','total_views','likes','comments',
            'points_sf','place_sf','LY_final_vote','LY_final_reciprocation'})
        self.assertTrue(all(r['core_training_ready']==1 for r in self.entries))
        self.assertTrue(all(r['audio_status']=='published_studio_features_accepted_by_user' for r in self.entries))

    def test_rebuild_is_byte_identical(self):
        second=Path(self.tmp.name)/'second'
        with contextlib.redirect_stdout(io.StringIO()):
            builder.build(second/'processed',second/'reports')
        for file in self.report.glob('*'):
            self.assertEqual(file.read_bytes(),(second/'reports'/file.name).read_bytes(),file.name)
        self.assertEqual((self.output/'entries.csv').read_bytes(),(second/'processed/entries.csv').read_bytes())

    def test_source_hash_failure_preserves_existing_output(self):
        sentinel=(self.output/'entries.csv').read_bytes()
        actual_sha=builder.sha256
        def tampered_hash(path):
            return 'bad-source-hash' if path==builder.SAMPLES/'finalists.csv' else actual_sha(path)
        with patch.object(builder,'sha256',side_effect=tampered_hash):
            with self.assertRaisesRegex(ValueError,'failed checks'):
                builder.build(self.output,Path(self.tmp.name)/'failed_reports')
        self.assertEqual((self.output/'entries.csv').read_bytes(),sentinel)


if __name__=='__main__':
    unittest.main()


In [ ]:
%%writefile {BUILD_DIR}/tests/test_final_inputs.py
"""Check complete pre-final evidence and restricted, matched market comparisons."""
import datetime as dt
import json
import math
import sys
import unittest
from pathlib import Path
from unittest.mock import patch

ROOT=Path(__file__).resolve().parents[1]
sys.path.insert(0,str(ROOT/'scripts'))
from build_entries import read_csv
from build_final_inputs import validate_running_orders, market_snapshot
from market_archive import archive_market, YEARS, PANEL

class FinalInputTests(unittest.TestCase):
    @classmethod
    def setUpClass(cls):
        cls.entries=read_csv(ROOT/'data/processed/entries.csv')

    def test_all_17_published_orders_match_exactly(self):
        rows=validate_running_orders(self.entries)
        self.assertEqual(len(rows),437)
        self.assertEqual(len({r['year'] for r in rows}),17)

    def test_changed_source_fails(self):
        with patch('build_final_inputs.sha256',return_value='tampered'):
            with self.assertRaisesRegex(ValueError,'source changed'):
                validate_running_orders(self.entries)

    def test_known_market_quotes_and_pre_cutoff_bound(self):
        expected={2023:26,2024:25,2025:26}
        for year,n in expected.items():
            rows=market_snapshot(year,self.entries)
            self.assertEqual(len(rows),n)
            self.assertAlmostEqual(sum(r['p_market'] for r in rows),1)
            for row in rows:
                cutoff=dt.datetime.fromisoformat(next(e['cutoff_utc'] for e in self.entries if int(e['year'])==year))
                self.assertLessEqual(dt.datetime.fromisoformat(row['captured_at_utc_upper']),cutoff)
                self.assertLessEqual(row['max_age_hours'],26)
                self.assertEqual(row['captured_at_utc'],'')
                self.assertAlmostEqual(row['market_strength'],math.log(row['p_market']))

    def test_future_odds_do_not_change_features(self):
        original=json.loads((ROOT/'data/raw/final_validation/odds2025_api.json').read_text())
        changed={k:v+[dict(x='2025-05-18 00:00',y=99)] for k,v in original.items()}
        baseline=market_snapshot(2025,self.entries)
        with patch('build_final_inputs.load_json',return_value=changed):
            self.assertEqual(market_snapshot(2025,self.entries),baseline)

    def test_missing_finalist_quote_fails_whole_year(self):
        original=json.loads((ROOT/'data/raw/final_validation/odds2025_api.json').read_text())
        original['AT']=[r for r in original['AT'] if r['x']!='2025-05-17 00:00']
        with patch('build_final_inputs.load_json',return_value=original):
            with self.assertRaisesRegex(ValueError,'unique simultaneous'):
                market_snapshot(2025,self.entries)

    def test_market_and_core_have_identical_complete_observations(self):
        sets=json.loads((ROOT/'data/processed/feature_sets.json').read_text())
        for task,count in [('A',259),('B',232),('C',8770)]:
            core=read_csv(ROOT/f'data/processed/model_{task}_core_market_years.csv')
            market=read_csv(ROOT/f'data/processed/model_{task}_market.csv')
            self.assertEqual(len(core),count)
            self.assertEqual(core,[{k:r[k] for k in core[0]} for r in market])
            self.assertEqual(sets[task+'_market'],sets[task]+['market_strength'])
            years=YEARS if task=='A' else [y for y in YEARS if y>=2016]
            self.assertEqual({int(r['year']) for r in market},set(years))
            if task=='C':
                from collections import defaultdict
                groups=defaultdict(list)
                for r in market:groups[(r['year'],r['voter'])].append(r)
                self.assertEqual(len(groups),349)
                for rows in groups.values():
                    self.assertEqual(len(rows),int(rows[0]['candidate_count']))
                    self.assertEqual(sum(int(r['target_first_place']) for r in rows),1)

    def test_fixed_archive_panel_complete_all_years(self):
        features,quotes=archive_market(self.entries)
        self.assertEqual(len(features),259)
        self.assertEqual(len(quotes),1295)
        for year in YEARS:
            rows=[r for r in features if r['year']==year]
            finalists={r['country'] for r in self.entries if int(r['year'])==year}
            self.assertEqual({r['country'] for r in rows},finalists)
            self.assertAlmostEqual(sum(r['p_market'] for r in rows),1)
            for row in rows:
                self.assertEqual(row['bookmaker_count'],5)
                self.assertEqual(row['bookmaker_panel'],';'.join(PANEL))
                self.assertEqual(row['timestamp_status'],'unknown_accepted_by_user')
                self.assertEqual(row['cutoff_verified'],0)
                self.assertEqual(row['captured_at_utc'],'')
                self.assertAlmostEqual(row['market_strength'],math.log(row['p_market']))

    def test_archive_normalizes_each_book_before_averaging(self):
        odds=[(2,2),(2,6),(4,2),(2,2),(2,2)]
        quotes=[dict(year=2025,country=c,bookmaker=b,decimal_odds=p)
            for b,pair in zip(PANEL,odds) for c,p in zip(['X','Y'],pair)]
        entries=[dict(year='2025',country=c) for c in ['X','Y']]
        with patch('market_archive.YEARS',[2025]),patch('market_archive.read_archive_tables',return_value=quotes):
            result,_=archive_market(entries)
        self.assertAlmostEqual(next(r['p_market'] for r in result if r['country']=='X'),31/60)

    def test_archive_missing_one_fixed_book_quote_fails(self):
        quotes=[dict(year=2025,country=c,bookmaker=b,decimal_odds=2)
            for b in PANEL for c in ['X','Y']][:-1]
        with patch('market_archive.YEARS',[2025]),patch('market_archive.read_archive_tables',return_value=quotes):
            with self.assertRaisesRegex(ValueError,'Incomplete finalist panel'):
                archive_market([dict(year='2025',country=c) for c in ['X','Y']])

    def test_archive_does_not_use_contest_outcomes(self):
        baseline,_=archive_market(self.entries)
        changed=[dict(r,place_final='1',points_final='9999',target_top10='1') for r in self.entries]
        modified,_=archive_market(changed)
        self.assertEqual(modified,baseline)

if __name__=='__main__':
    unittest.main()


In [ ]:
%%writefile {BUILD_DIR}/tests/test_remaining.py
"""Behavioral tests for chronological features, complete choice sets and geography."""
import collections
import contextlib
import copy
import io
import math
from pathlib import Path
import sys
import tempfile
import unittest
from unittest.mock import patch

sys.path.insert(0,str(Path(__file__).resolve().parents[1]/'scripts'))
import build_remaining as b


class HistoryTests(unittest.TestCase):
    def test_neutral_priors_with_no_history(self):
        e=dict(year=2016,country='AA')
        self.assertEqual(b.country_history([],e),(0.5,0.0,[]))
        self.assertEqual(b.country_history([],e,start=2016,kind='split'),(0.0,0.0,[]))
        self.assertEqual(b.pair_history({},'AA','BB',2016),(0.0,0.0,[]))

    def test_A_exact_formula(self):
        rows=[dict(year=2018,country='AA',n_finalists=26,place_final=1),
              dict(year=2019,country='AA',n_finalists=26,place_final=26)]
        value,count,selected=b.country_history(rows,dict(year=2021,country='AA'))
        self.assertEqual(value,(1+0+1.5)/5)
        self.assertEqual(count,math.log(3))
        self.assertEqual(len(selected),2)

    def test_window_is_calendar_years_not_last_five_contests(self):
        rows=[dict(year=y,country='AA',n_finalists=26,place_final=1) for y in range(2015,2023)]
        _,_,selected=b.country_history(rows,dict(year=2021,country='AA'))
        self.assertEqual([r['year'] for r in selected],[2016,2017,2018,2019])

    def test_B_uses_rank_difference_not_point_difference(self):
        rows=[dict(year=2019,country='AA',n_finalists=26,place_jury=16,place_tele=6,
            points_jury=1000,points_tele=1)]
        value,count,_=b.country_history(rows,dict(year=2021,country='AA'),kind='split',start=2016)
        self.assertAlmostEqual(value,(10/25)/4)
        self.assertEqual(count,math.log(2))

    def test_pair_zero_is_counted_and_absence_is_not(self):
        rows=[dict(year=2018,public_points=0,candidate_count=25),dict(year=2019,public_points=12,candidate_count=26)]
        index={('AA','BB'):rows}
        value,count,selected=b.pair_history(index,'AA','BB',2021)
        self.assertAlmostEqual(value,(-58/25+12-58/26)/5)
        self.assertEqual(count,math.log(3))
        self.assertEqual(len(selected),2)
        self.assertEqual(b.pair_history(index,'BB','AA',2021),(0,0,[]))

    def test_pair_history_does_not_expand_across_cancelled_year(self):
        index={('AA','BB'):[dict(year=y,public_points=0,candidate_count=25) for y in [2015,2016,2019,2020,2021,2022]]}
        _,_,selected=b.pair_history(index,'AA','BB',2021)
        self.assertEqual([r['year'] for r in selected],[2016,2019])

    def test_future_and_current_outcomes_cannot_change_features(self):
        rows=[dict(year=y,country='AA',n_finalists=26,place_final=10) for y in range(2003,2026) if y!=2020]
        e=dict(year=2013,country='AA');before=b.country_history(rows,e)
        changed=copy.deepcopy(rows)
        for row in changed:
            if row['year']>=2013:row['place_final']=1
        self.assertEqual(before,b.country_history(changed,e))

    def test_predecessor_not_inherited(self):
        rows=[dict(year=2005,country='CS',n_finalists=24,place_final=1)]
        self.assertEqual(b.country_history(rows,dict(year=2008,country='RS')),(0.5,0.0,[]))


class GeographyTests(unittest.TestCase):
    def test_great_circle_known_arc(self):
        self.assertAlmostEqual(b.great_circle(0,0,0,90),math.pi*6371.0088/2)
        self.assertEqual(b.great_circle(45,20,45,20),0)
        self.assertAlmostEqual(b.great_circle(52,13,48,2),b.great_circle(48,2,52,13))


class PreparationTests(unittest.TestCase):
    @classmethod
    def setUpClass(cls):
        cls.tmp=tempfile.TemporaryDirectory();cls.output=Path(cls.tmp.name)/'output';cls.report=Path(cls.tmp.name)/'report'
        with contextlib.redirect_stdout(io.StringIO()):cls.tables=b.build(cls.output,cls.report)
        cls.entries={(r['year'],r['country']):r for r in cls.tables['entries_features.csv']}
        cls.pairs={(r['voter'],r['recipient']):r for r in cls.tables['country_pairs.csv']}

    @classmethod
    def tearDownClass(cls):cls.tmp.cleanup()

    def test_model_sizes_and_predictor_counts(self):
        self.assertEqual([len(self.tables[f'model_{task}.csv']) for task in 'ABC'],[437,232,8770])
        self.assertEqual([len(x) for x in [b.FEATURES_A,b.FEATURES_B,b.FEATURES_C]],[11,11,13])

    def test_country_history_has_early_sources(self):
        row=self.entries[2008,'RU']
        self.assertEqual(row['past_final_n'],5)
        self.assertEqual(row['past_final_source_years'],'2003;2004;2005;2006;2007')
        self.assertEqual(self.entries[2016,'SE']['past_split_n'],0)

    def test_national_groups_and_substitutes(self):
        votes=self.tables['national_votes.csv']
        groups=collections.defaultdict(list)
        for r in votes:groups[r['year'],r['voter']].append(r)
        self.assertEqual(len(groups),358)
        self.assertEqual(sum(g[0]['eligible_C'] for g in groups.values()),349)
        for (year,voter),rows in groups.items():
            self.assertEqual(len(rows),rows[0]['candidate_count'])
            self.assertFalse(any(r['recipient']==voter for r in rows))
            self.assertEqual(sum(r['public_points'] for r in rows),58)
            if voter=='SM':
                self.assertTrue(all(r['eligible_C']==0 and r['target_first_place'] is None for r in rows))
                self.assertTrue(all(r['past_pair_n']==0 for r in rows))
            else:self.assertEqual(sum(r['target_first_place'] for r in rows),1)

    def test_jury_replacement_does_not_remove_real_televote(self):
        voters={r['voter'] for r in self.tables['model_C.csv'] if r['year']==2022}
        self.assertTrue({'AZ','GE','ME','PL','RO'}<=voters)
        self.assertNotIn('SM',voters)

    def test_nonfinalist_voters_preserved(self):
        final={e['country'] for e in self.tables['entries_features.csv'] if e['year']==2024}
        voters={r['voter'] for r in self.tables['model_C.csv'] if r['year']==2024}
        self.assertNotIn('NL',final)
        self.assertIn('NL',voters)

    def test_row_separate_but_overall_points_unchanged(self):
        self.assertEqual(len(self.tables['non_national_votes.csv']),77)
        self.assertFalse(any(r['voter']=='WLD' for r in self.tables['national_votes.csv']))
        self.assertEqual(self.entries[2025,'AT']['points_tele'],178)

    def test_real_zero_points_contribute_to_pair_history(self):
        votes=self.tables['national_votes.csv']
        lookup={(r['year'],r['voter'],r['recipient']):r for r in votes}
        candidates=[r for r in votes if r['year']==2025 and r['eligible_C'] and r['past_pair_n']>0]
        tested=False
        for row in candidates:
            history=[lookup[y,row['voter'],row['recipient']] for y in range(2020,2025)
                if (y,row['voter'],row['recipient']) in lookup and lookup[y,row['voter'],row['recipient']]['eligible_C']]
            if any(r['public_points']==0 for r in history):
                self.assertEqual(row['past_pair_n'],len(history))
                self.assertAlmostEqual(row['past_pair_support'],sum(r['public_points']-58/r['candidate_count'] for r in history)/(len(history)+3))
                tested=True;break
        self.assertTrue(tested)

    def test_geography_all_pairs_and_edge_cases(self):
        self.assertEqual(len(self.pairs),46*45)
        for a,b,expected in [('GB','IE',1),('IT','SM',1),('AZ','TR',1),('FR','NL',0),('GB','ES',0),('GB','CY',0),('DK','SE',0)]:
            self.assertEqual(self.pairs[a,b]['shared_border'],expected)
        self.assertTrue(all(r['shared_border']==0 for r in self.pairs.values() if r['voter']=='AU'))

    def test_no_leakage_columns_in_predictor_sets(self):
        forbidden={'year','country','voter','recipient','public_points','target_first_place',
            'target_top10','target_audience_above_jury','place_final','points_final','past_pair_n'}
        for fields in [b.FEATURES_A,b.FEATURES_B,b.FEATURES_C]:self.assertFalse(forbidden & set(fields))

    def test_rebuild_is_byte_identical(self):
        second=Path(self.tmp.name)/'second'
        with contextlib.redirect_stdout(io.StringIO()):b.build(second/'output',second/'report')
        for file in self.output.glob('*'):self.assertEqual(file.read_bytes(),(second/'output'/file.name).read_bytes(),file.name)
        for file in self.report.glob('*'):self.assertEqual(file.read_bytes(),(second/'report'/file.name).read_bytes(),file.name)

    def test_bad_source_preserves_outputs(self):
        before=(self.output/'model_C.csv').read_bytes();real=b.sha256
        def tampered(path):return 'bad' if path==b.RAW/'geography/ne_10m_populated_places.geojson' else real(path)
        with patch.object(b,'sha256',side_effect=tampered):
            with self.assertRaisesRegex(ValueError,'source_hash'):b.build(self.output,Path(self.tmp.name)/'failed')
        self.assertEqual((self.output/'model_C.csv').read_bytes(),before)


if __name__=='__main__':unittest.main()


In [ ]:
test_run = subprocess.run(
    [sys.executable, "-m", "unittest", "discover", "-s", "tests", "-v"],
    cwd=BUILD_DIR, capture_output=True, text=True,
)
(BUILD_DIR / "acceptance_tests.log").write_text(test_run.stdout + test_run.stderr, encoding="utf-8")
print(test_run.stderr)
if test_run.returncode:
    raise RuntimeError("Проверки не пройдены. См. acceptance_tests.log")

## 7. Excel для модельных ноутбуков

Создаём **новую** книгу из пересобранных CSV. Шесть листов и порядок столбцов
соответствуют `eurovision_features.xlsx`, использованному в A/B/C.
Числа остаются числовыми ячейками; добавляем фильтры и закрепляем заголовок.
Проверяем повторное чтение книги, включая все значения и названия столбцов.

In [ ]:
SHEETS = {
    "A Топ-10": "A",
    "B Зрители и жюри": "B",
    "C Национальное голосование": "C",
    "A С рынком": "A_market",
    "B С рынком": "B_market",
    "C С рынком": "C_market",
}
XLSX_PATH = BUILD_DIR / "eurovision_features.xlsx"
from openpyxl.styles import Font, PatternFill, Alignment
with pd.ExcelWriter(XLSX_PATH, engine="openpyxl") as writer:
    for sheet_name, task in SHEETS.items():
        frame = tables[task]
        frame.to_excel(writer, sheet_name=sheet_name, index=False)
        ws = writer.sheets[sheet_name]
        ws.freeze_panes = "C2"
        ws.auto_filter.ref = ws.dimensions
        ws.row_dimensions[1].height = 34
        for cell in ws[1]:
            cell.fill = PatternFill("solid", fgColor="203864")
            cell.font = Font(color="FFFFFF", bold=True)
            cell.alignment = Alignment(wrap_text=True, vertical="center")
        for col in ws.columns:
            ws.column_dimensions[col[0].column_letter].width = min(38, max(13, len(str(col[0].value)) + 2))
with pd.ExcelFile(XLSX_PATH) as workbook:
    assert workbook.sheet_names == list(SHEETS)
    for sheet_name, task in SHEETS.items():
        pd.testing.assert_frame_equal(workbook.parse(sheet_name), tables[task],
                                      check_dtype=False, check_exact=False, rtol=1e-12, atol=1e-12)
print("Excel создан и проверен:", XLSX_PATH)
display(FileLink(str(XLSX_PATH)))

## 8. Журнал воспроизведения и результаты

В выгрузку входят Excel, CSV, разрешённые списки признаков, происхождение,
отчёты проверки и журнал получения исходников. Модели здесь не обучаются.
В модельном ноутбуке достаточно указать путь к полученному Excel.

In [ ]:
verification = dict(
    python=platform.python_version(), pandas=pd.__version__, openpyxl=openpyxl.__version__,
    source_mode=SOURCE_MODE, raw_files_verified=len(manifest["files"]),
    source_manifest_sha256=file_sha256(SOURCE_DIR / "reproduction_manifest.json"),
    output_csv_matches_original=bool(csv_checks.matches_original.all()),
    csv_count=len(csv_checks), acceptance_tests_passed=test_run.returncode == 0,
    sheet_rows={name: len(tables[task]) for name, task in SHEETS.items()},
    market_snapshot_time="unknown", market_cutoff_verified=False,
    output_csv_sha256={p.name: file_sha256(p) for p in sorted(PROCESSED.glob("*.csv"))},
    feature_sets=feature_sets,
)
(BUILD_DIR / "reproduction_verification.json").write_text(
    json.dumps(verification, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")
shutil.copyfile(BUILD_DIR / "reproduction_manifest.json", BUILD_DIR / "reports/source_manifest.json")
bundle = BUILD_DIR / "eurovision_rebuilt_outputs.zip"
with zipfile.ZipFile(bundle, "w", compression=zipfile.ZIP_DEFLATED) as archive:
    for folder in [PROCESSED, BUILD_DIR / "reports"]:
        for path in sorted(folder.rglob("*")):
            if path.is_file():
                archive.write(path, str(path.relative_to(BUILD_DIR)))
    for name in ["eurovision_features.xlsx", "source_acquisition.csv", "acceptance_tests.log",
                 "reproduction_verification.json"]:
        archive.write(BUILD_DIR / name, name)
print("Готово: исходники распарсены, таблицы совпали с данными экспериментов, Excel проверен.")
display(FileLink(str(bundle)))